# SmartMentor · Fase 0 · Prueba «Mago de Oz» con Gemini (v2)

Este notebook es **el mago detrás de la cortina**: antes de construir la plataforma, el equipo le pide al LLM, a mano y con prompts versionados, todo lo que las pantallas van a mostrar. Así se comprueba que la IA produce algo útil y se fija el **contrato de datos** entre la IA y la interfaz.

## Qué pide el brief

En la Fase 0 del [brief](../../proyecto/brief.md#fase-0-prototipado-de-la-interfaz-y-del-flujo), la actividad 4 es:

> Prueba de prompts tipo «Mago de Oz»: el equipo genera a mano con el LLM el syllabus estructurado, el diagnóstico y un tema, y los muestra en el prototipo. **Herramienta:** notebook + LLM. **Resultado:** primeros ejemplos reales de salida y esquemas JSON.

Y el criterio de salida de la fase:

> Flujo y pantallas aprobados, **esquemas JSON de cada salida de IA definidos y versionados en el repositorio**.

## Dónde se cumple cada exigencia

| Exigencia del brief | Sección | Evidencia que queda |
| --- | --- | --- |
| Syllabus estructurado generado con el LLM | §6 | `outputs/syllabus.json` |
| Diagnóstico generado con el LLM | §7 | `outputs/diagnostic.json` |
| Un tema completo (texto + video + quiz) | §10 | `outputs/topic_explanation_*.json`, `video_*.json`, `quiz_*.json` |
| Mostrarlo en el prototipo | §14 | `outputs/screens_in_progress.json` y `screens_final.json`, con las claves de las pantallas de Figma |
| Primeros ejemplos **reales** de salida | Todo el notebook | Este notebook **guardado con sus salidas** + `llm_log.jsonl` |
| Esquemas JSON de cada salida | §2 y §15 | `schemas/*.v2.json` |
| Esquemas versionados en el repositorio | §17 | Paquete listo para copiar a `prompts/` |

El notebook va más allá del mínimo (syllabus, diagnóstico y un tema): recorre la ruta completa de una estudiante simulada, porque así cada pantalla del prototipo se llena desde **una sola fuente de datos** y desaparecen las inconsistencias que encontró el análisis del Figma (6 contra 5 temas, tres puntajes distintos para el mismo tema, badge antes de terminar la ruta).

## Cómo encaja con el resto del brief

| Componente de IA del brief | Sección | Cómo se implementa aquí |
| --- | --- | --- |
| 1. Estructuración del syllabus (JSON validado, DAG revisado por el docente) | §6 | Prompt + esquema + verificación de ciclos + celda de correcciones del docente |
| 2. Prueba diagnóstica (2 a 3 preguntas por tema en 3 niveles de Bloom, nivel ponderado por dificultad) | §7 | 3 preguntas por tema (remember, apply, analyze), revisión automática de ítems |
| 3. Perfil semántico (embeddings del perfil contra cada tema, la prueba pesa más) | §7 | Similitud coseno, peso máximo del perfil 25 % y menor si la persona escribe poco |
| 4. Objetivo de aprendizaje (verbo de Bloom + contenido + criterio) | §8 | Prompt con la brecha; el estudiante lo confirma |
| 5. Secuenciación (orden topológico, omitir ≥ 80 %) | §9 | Algoritmo determinista, sin LLM |
| 6. Recursos: texto con RAG citando fuentes y video re-ordenado por similitud | §10 | Recuperación por embeddings, verificación de fidelidad, re-ranking de YouTube |
| 7. Evaluaciones desde los resultados de aprendizaje y LLM-as-judge con rúbrica | §10, §11, §12 | Quiz y final sin texto de estudio en el prompt; calificador con rúbrica explícita |

| Decisión técnica recomendada por el brief | Dónde |
| --- | --- |
| Salidas estructuradas con esquemas | §2 y §4: `response_json_schema` + validación con Pydantic + reintento |
| Versionar los prompts como código | §3: cada prompt tiene versión y viaja en `meta` de cada salida |
| Caché del contenido por tema, sin regenerar por estudiante | §4 y §10: la explicación principal es un recurso compartido; solo el ejemplo y la explicación de refuerzo son personales |
| Registrar cada llamada (prompt, modelo, tokens, costo, latencia) | §4: `llm_log.jsonl` con los mismos campos que la tabla `llm_logs` de la API |

## Cambios frente a la v1

| Cambio | Por qué |
| --- | --- |
| La explicación principal del tema ya no recibe el perfil ni el nivel del estudiante | El brief pide caché por tema y un recurso compartido entre estudiantes. La personalización pasa a un ejemplo corto (`personalized_example`) y a la explicación de refuerzo (`remedial_explanation`) |
| Campos de los esquemas en inglés y alineados con la API (`bloom_level`, `learning_outcomes`, `prerequisites`, estados `pending`/`passed`/`skipped`, umbrales de 0 a 100) | Regla del repositorio (código y JSON en inglés) y para que la API y la app reciban el contrato sin traducir |
| Nueva revisión automática de ítems (`item_review`) y exportación para revisión de expertos (§13) | El brief mitiga las preguntas ambiguas con un segundo prompt y mide su calidad con 2 expertos (kappa de Cohen) y el calificador con QWK |
| El quiz ya no recibe fragmentos del texto | El brief pide generar las evaluaciones desde los resultados de aprendizaje, no desde el recurso |
| «No lo sé» en el diagnóstico | Existe en la pantalla E3 y el estudiante simulado no podía usarlo |
| `estimated_hours` = 0 si el syllabus no lo dice | La v1 obligaba al LLM a inventar horas, contra la regla de no inventar |
| Clasificación del diagnóstico (dominado, por reforzar, por aprender) | Cortes propuestos en los casos de uso del diseño, que la pantalla E4 usa sin definir |
| El repaso de la evaluación final solo usa temas de la ruta | En la v1 fallaba si una pregunta caía en un tema omitido |

## Cómo usarlo

1. Pon una API key de Gemini en `GEMINI_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey); en Colab: *Secrets*). Opcional: `YOUTUBE_API_KEY`.
2. Cambia el syllabus de ejemplo por uno real del piloto (§5) y llena los precios del modelo (§1).
3. Ejecuta todo y **guarda el notebook con sus salidas**: es la evidencia de que el contenido salió del LLM y no se redactó a mano.
4. Pide a dos expertos que llenen `review/items_for_experts.csv` y vuelve a correr §13.
5. Copia el paquete de §17 al repositorio.

**Qué no es.** Las respuestas de «Laura» las produce un estudiante simulado con LLM (el brief lo acepta como respaldo). Sirven para recorrer el flujo y llenar el prototipo, no para medir aprendizaje: eso es el piloto, y la usabilidad se mide en la actividad 5 con personas reales.

In [ ]:
%pip install -q -U google-genai pydantic jsonschema numpy pandas requests

## 1. Configuración

Los umbrales son las reglas de negocio del brief («aprobar un tema con ≥ 70 %, máximo 3 intentos, evaluación final con ≥ 80 % para el badge»; «se omiten los temas con dominio ≥ 80 %»). El docente los define por ruta, así que aquí se usan los mismos nombres y la misma escala de 0 a 100 que los campos de `learning_paths` en la API.

El corte de 50 entre «por reforzar» y «por aprender» es una **propuesta** del diseño (decisión 4 de [`casos-de-uso.md`](../01-flujo-y-casos-de-uso/casos-de-uso.md#decisiones-que-el-director-debe-tomar)) que el director debe aprobar.

Llena `PRICES_USD_PER_MILLION` con la tarifa vigente ([precios de Gemini](https://ai.google.dev/gemini-api/docs/pricing)); sin eso el costo queda nulo y no se puede reportar la métrica de costo por estudiante-ruta.

In [ ]:
import os, json, time, hashlib, datetime as dt, pathlib, re, copy, textwrap, shutil
from typing import Literal, Optional
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from google import genai
from google.genai import types

try:
    display
except NameError:  # fuera de Jupyter
    display = print

# --- API key: variable de entorno o Colab Secrets ---
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
assert os.environ.get("GEMINI_API_KEY"), "Define GEMINI_API_KEY (variable de entorno o Colab Secrets)."
client = genai.Client()

# --- Modelos ---
LLM_MODEL = os.environ.get("SM_LLM_MODEL", "gemini-3.5-flash")        # generación (confírmalo en la celda siguiente)
EMBED_MODEL = os.environ.get("SM_EMBED_MODEL", "gemini-embedding-001")  # embeddings de texto
EMBED_DIM = 768
TEMPERATURE = 0.4

# --- Reglas de la ruta (0-100, como learning_paths en la API) ---
TOPIC_PASS_THRESHOLD = 70     # aprobar el quiz de un tema
FINAL_PASS_THRESHOLD = 80     # evaluación final → badge
MASTERY_SKIP_THRESHOLD = 80   # tema omitido si el diagnóstico ya muestra dominio
MAX_ATTEMPTS = 3
REINFORCE_CUTOFF = 50         # propuesta: < 50 por aprender, 50-79 por reforzar, ≥ 80 dominado

# --- Precios (USD por millón de tokens) para el costo del log ---
PRICES_USD_PER_MILLION = {
    # "gemini-3.5-flash": {"input": 0.0, "output": 0.0},
    # "gemini-embedding-001": {"input": 0.0, "output": 0.0},
}

REGENERATE = False  # True = ignora la caché y vuelve a llamar al LLM

OUT = pathlib.Path("mago_de_oz_out_v2")
for folder in ["schemas", "prompts", "outputs", "cache", "review"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT / "llm_log.jsonl"
print("LLM:", LLM_MODEL, "| Embeddings:", EMBED_MODEL, "| salida en", OUT.resolve())

In [ ]:
# Modelos disponibles para tu clave: confirma que LLM_MODEL aparezca en la lista
try:
    rows = []
    for m in client.models.list():
        actions = getattr(m, "supported_actions", None) or []
        if "generateContent" in actions or "embedContent" in actions:
            rows.append({"model": m.name.replace("models/", ""), "actions": ", ".join(actions)})
    available = pd.DataFrame(rows)
    display(available)
    if not available.empty and LLM_MODEL not in set(available["model"]):
        print(f"⚠️ {LLM_MODEL} no aparece en la lista: cambia SM_LLM_MODEL.")
except Exception as e:
    print("No se pudo listar modelos:", e)

## 2. Contratos de datos (esquemas JSON)

Este es **el entregable principal** de la actividad y del criterio de salida. Cada salida de IA tiene dos capas:

- **Modelo `...LLM`**: lo que se le exige devolver al LLM. Se pasa a Gemini como `response_json_schema`. Evita uniones y diccionarios libres, que Gemini respeta peor; los rangos se validan en código.
- **Contrato**: el modelo LLM más los datos de trazabilidad (`meta` con prompt, versión, modelo y hash de la entrada). Es lo que guarda el backend y lo que recibe la pantalla.

Los nombres de los campos están en inglés y siguen a la API: `learning_outcomes`, `bloom_level`, `prerequisites` (tabla `topics`), `default_objective` (`learning_paths`), `profile_description` y `personal_objective` (`enrollments`), `level_by_topic` (`diagnostics`), `step_order`, `status` y `score` (`path_steps`), `assessment_type` (`assessment_attempts`), `open_badges_metadata` (`badges`). El contenido que lee el estudiante (enunciados, explicaciones) va en español.

| Contrato (`schemas/<nombre>.v2.json`) | Salida del brief | Entidad de la API | Pantallas |
| --- | --- | --- | --- |
| `syllabus` | Syllabus estructurado | `courses`, `topics`, `topic_prerequisites` | D2 |
| `diagnostic` | Diagnóstico | `diagnostics` | E3, E4 |
| `objective` | Objetivo de aprendizaje | `enrollments.personal_objective` | E4 |
| `learning_path` | Ruta personalizada | `path_steps` | E5, D3 |
| `topic_explanation` | Texto del tema (RAG) | `resources` (`text`) | E6 |
| `personalized_example` | Ejemplo conectado con el perfil | — (por inscripción) | E6 |
| `video_resource` | Video del tema | `resources` (`video`) | E6 |
| `quiz` | Quiz del tema | `assessment_attempts` (`topic`) | E6 |
| `grading` | Calificación (regla o LLM-as-judge) | `assessment_attempts.score` | E7, D4 |
| `final_assessment` | Evaluación final | `assessment_attempts` (`final`) | E8, E9 |
| `badge` | Badge Open Badges 2.0 | `badges.open_badges_metadata` | E10, E12 |

In [ ]:
BloomLevel = Literal["remember", "understand", "apply", "analyze", "evaluate", "create"]
BLOOM_ORDER = ["remember", "understand", "apply", "analyze", "evaluate", "create"]
BLOOM_LABEL_ES = {"remember": "Recordar", "understand": "Comprender", "apply": "Aplicar",
                  "analyze": "Analizar", "evaluate": "Evaluar", "create": "Crear"}

class Meta(BaseModel):
    prompt_id: str
    prompt_version: str
    model: str
    generated_at: str
    input_sha256: str
    from_cache: bool = False

# ---------- Syllabus ----------
class LearningOutcome(BaseModel):
    id: str = Field(description="lo_<topic_id>_<n>")
    statement: str = Field(description="Verbo de Bloom + contenido + criterio medible, en español")
    bloom_level: BloomLevel

class SyllabusTopic(BaseModel):
    id: str = Field(description="snake_case estable, ej. regresion_logistica")
    name: str
    description: str
    subtopics: list[str]
    learning_outcomes: list[LearningOutcome]
    bloom_level: BloomLevel = Field(description="El nivel más alto entre sus resultados de aprendizaje")
    prerequisites: list[str] = Field(description="ids de temas de este mismo syllabus")
    estimated_hours: int = Field(description="Horas indicadas en el syllabus; 0 si no las indica")

class SyllabusLLM(BaseModel):
    course_id: str
    course_name: str
    default_objective: str
    topics: list[SyllabusTopic]

class Syllabus(SyllabusLLM):
    approved_by_teacher: bool = False
    teacher_corrections: list[str] = []
    meta: Meta

# ---------- Revisión automática de ítems ----------
class ItemReview(BaseModel):
    question_id: str
    single_correct_answer: bool
    clear_stem: bool
    bloom_matches: bool
    aligned_with_outcome: bool
    issues: list[str]

class ItemReviewLLM(BaseModel):
    reviews: list[ItemReview]

# ---------- Diagnóstico ----------
class DiagnosticQuestion(BaseModel):
    id: str
    topic_id: str
    stem: str
    bloom_level: BloomLevel
    difficulty: int = Field(description="1 = remember, 2 = apply, 3 = analyze")
    options: list[str]
    correct_option: int = Field(description="Índice 0-based de la opción correcta")
    rationale: str

class DiagnosticLLM(BaseModel):
    questions: list[DiagnosticQuestion]

class StudentAnswer(BaseModel):
    question_id: str
    selected_option: int = Field(description="Índice 0-based; -1 en respuesta corta o «No lo sé»")
    dont_know: bool
    text: str = Field(description="Respuesta escrita; vacío en opción múltiple")

class SimulatedAnswers(BaseModel):
    answers: list[StudentAnswer]

class TopicLevel(BaseModel):
    topic_id: str
    test_score: float          # 0-100, aciertos ponderados por dificultad
    profile_similarity: float  # coseno perfil vs tema
    profile_prior: float       # 0-100, similitud normalizada
    profile_weight: float      # 0-1, cuánto pesa el perfil
    level: float               # 0-100, nivel combinado
    classification: Literal["mastered", "to_reinforce", "to_learn"]
    skip: bool

class Diagnostic(BaseModel):
    enrollment_id: str
    questions: list[DiagnosticQuestion]
    item_reviews: list[ItemReview]
    discarded_questions: list[str]
    answers: list[StudentAnswer]
    level_by_topic: list[TopicLevel]
    overall_score: float
    meta_questions: Meta
    meta_answers: Meta

# ---------- Objetivo ----------
class ObjectiveLLM(BaseModel):
    statement: str = Field(description="Verbo de Bloom + contenido + criterio, en español")
    bloom_verb: str
    content: str
    criterion: str
    focus_topics: list[str] = Field(description="ids de los temas de la brecha")
    rationale: str

class Objective(ObjectiveLLM):
    enrollment_id: str
    origin: Literal["path_default", "llm_proposed"]
    confirmed_by_student: bool
    meta: Optional[Meta] = None

# ---------- Ruta ----------
StepStatus = Literal["pending", "in_progress", "passed", "failed", "skipped"]  # igual que la API

class PathStep(BaseModel):
    step_order: int
    topic_id: str
    name: str
    included: bool
    reason: str
    status: StepStatus
    score: Optional[float] = None   # 0-100
    attempts: int = 0

class PersonalizedPath(BaseModel):
    enrollment_id: str
    learning_path_id: str
    algorithm: str
    mastery_skip_threshold: float
    steps: list[PathStep]

# ---------- Explicación del tema (RAG) ----------
class Section(BaseModel):
    heading: str
    body: str = Field(description="Texto con citas [n] a los fragmentos")

class TopicExplanationLLM(BaseModel):
    title: str
    sections: list[Section]
    worked_example: str
    summary: str
    reflection_questions: list[str]
    gaps_detected: list[str] = Field(description="Resultados de aprendizaje que los fragmentos no cubren")

class CitedSource(BaseModel):
    n: int
    source_id: str
    title: str
    url: str

class TopicExplanation(TopicExplanationLLM):
    topic_id: str
    kind: Literal["main", "remedial"]
    audience: Literal["all_students", "enrollment"]   # main = recurso compartido; remedial = personal
    enrollment_id: Optional[str] = None
    sources: list[CitedSource]
    faithfulness: Optional[float] = None               # 0-1, afirmaciones soportadas / total
    unsupported_claims: list[str] = []
    meta: Meta

class Claim(BaseModel):
    text: str
    citations: list[int]
    supported: bool
    comment: str

class FaithfulnessCheck(BaseModel):
    claims: list[Claim]

class PersonalizedExampleLLM(BaseModel):
    example: str
    connection: str

class PersonalizedExample(PersonalizedExampleLLM):
    topic_id: str
    enrollment_id: str
    meta: Meta

# ---------- Video ----------
class VideoCandidate(BaseModel):
    video_id: str
    title: str
    url: str
    channel: str = ""
    duration_min: Optional[float] = None
    similarity: Optional[float] = None

class VideoResource(BaseModel):
    topic_id: str
    origin: Literal["youtube_api", "teacher_list", "none"]
    candidates: list[VideoCandidate]
    selected: Optional[VideoCandidate] = None
    embed_model: str

# ---------- Quiz y evaluación final ----------
class RubricCriterion(BaseModel):
    criterion: str
    points: float

class QuizQuestion(BaseModel):
    id: str
    topic_id: str
    outcome_id: str = Field(description="Resultado de aprendizaje que evalúa")
    stem: str
    type: Literal["multiple_choice", "short_answer"]
    bloom_level: BloomLevel
    options: list[str] = Field(description="Vacío en respuesta corta")
    correct_option: int = Field(description="Índice 0-based; -1 en respuesta corta")
    model_answer: str = Field(description="Justificación (opción múltiple) o respuesta esperada (respuesta corta)")
    rubric: list[RubricCriterion] = Field(description="Solo respuesta corta; vacío en opción múltiple")

class QuizLLM(BaseModel):
    questions: list[QuizQuestion]

class Quiz(QuizLLM):
    quiz_id: str
    topic_id: str
    attempt: int
    pass_threshold: float
    item_reviews: list[ItemReview]
    discarded_questions: list[str]
    meta: Meta

class FinalAssessment(QuizLLM):
    assessment_id: str
    topics_assessed: list[str]
    attempt: int
    pass_threshold: float
    item_reviews: list[ItemReview]
    discarded_questions: list[str]
    meta: Meta

# ---------- Calificación ----------
class CriterionResult(BaseModel):
    criterion: str
    met: bool
    points_awarded: float
    evidence: str

class ShortAnswerGradingLLM(BaseModel):
    criteria: list[CriterionResult]
    feedback: str

class QuestionResult(BaseModel):
    question_id: str
    topic_id: str
    type: str
    correct: Optional[bool] = None
    score: float                                  # 0-100
    grader: Literal["rule", "llm_judge"]
    feedback: str
    rubric_detail: list[CriterionResult] = []

class Grading(BaseModel):
    attempt_id: str
    assessment_id: str
    assessment_type: Literal["topic", "final"]    # igual que la API
    topic_id: Optional[str] = None
    answers: list[StudentAnswer]
    results: list[QuestionResult]
    score: float                                  # 0-100
    passed: bool
    submitted_at: str
    meta_answers: Meta
    meta_grader: list[Meta]

# ---------- Badge (Open Badges 2.0) ----------
class BadgeTextLLM(BaseModel):
    name: str
    description: str
    criteria: str
    competencies: list[str]

class OBCriteria(BaseModel):
    narrative: str

class OBIssuer(BaseModel):
    type: str = "Profile"
    id: str
    name: str
    url: str

class OBBadgeClass(BaseModel):
    type: str = "BadgeClass"
    id: str
    name: str
    description: str
    image: str
    criteria: OBCriteria
    issuer: OBIssuer
    tags: list[str]

class OBRecipient(BaseModel):
    type: str = "email"
    hashed: bool = True
    salt: str
    identity: str

class OBVerification(BaseModel):
    type: str = "hosted"

class OBEvidence(BaseModel):
    narrative: str

class BadgeAssertion(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    context: str = Field("https://w3id.org/openbadges/v2", alias="@context")
    type: str = "Assertion"
    id: str
    recipient: OBRecipient
    badge: OBBadgeClass
    issuedOn: str
    verification: OBVerification = OBVerification()
    evidence: list[OBEvidence]

# ---------- Exportación de los contratos ----------
SCHEMA_VERSION = "v2"
CONTRACTS = {
    "syllabus": Syllabus, "diagnostic": Diagnostic, "objective": Objective,
    "learning_path": PersonalizedPath, "topic_explanation": TopicExplanation,
    "personalized_example": PersonalizedExample, "video_resource": VideoResource,
    "quiz": Quiz, "grading": Grading, "final_assessment": FinalAssessment, "badge": BadgeAssertion,
}
for name, model in CONTRACTS.items():
    schema = model.model_json_schema(by_alias=True)
    schema["$id"] = f"smartmentor/schemas/{name}.{SCHEMA_VERSION}.json"
    (OUT / "schemas" / f"{name}.{SCHEMA_VERSION}.json").write_text(
        json.dumps(schema, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(CONTRACTS), "esquemas exportados:", ", ".join(sorted(CONTRACTS)))

## 3. Prompts versionados

Regla: **cambiar un prompt es subir su versión.** La versión viaja en `meta` de cada salida y en el log, así cada pantalla puede mostrar qué prompt produjo su contenido (como «explicacion_tema v1.3» en E6) y el análisis final puede comparar versiones.

Todos los prompts de la v1 pasan a `v2.0` porque su contrato de salida cambió (campos en inglés). Los nuevos empiezan en `v1.0`. Los ids de los prompts están en inglés; el texto, en español.

| Prompt | Componente del brief | Alcance | Cambio en v2 |
| --- | --- | --- | --- |
| `syllabus_structuring` | 1. Estructuración del syllabus | Por curso | No inventar horas |
| `diagnostic_generation` | 2. Prueba diagnóstica | Por curso | Justificación de cada distractor |
| `item_review` | Mitigación de preguntas ambiguas | Por ítem | **Nuevo** |
| `simulated_student` | Respaldo del piloto («estudiantes simulados con LLM») | Por inscripción | Opción «No lo sé» |
| `learning_objective` | 4. Objetivo de aprendizaje | Por inscripción | — |
| `topic_explanation` | 6. Texto con RAG | **Por tema (compartido)** | Sin perfil ni nivel |
| `remedial_explanation` | Bucle por tema: explicación alterna | Por inscripción | **Nuevo** (antes era un modo de la explicación) |
| `personalized_example` | Personalización del recurso | Por inscripción | **Nuevo** |
| `rag_faithfulness` | Métrica de fidelidad (tipo RAGAS) | Por explicación | — |
| `topic_quiz` | 7. Evaluaciones | Por tema e intento | Sin fragmentos del texto |
| `short_answer_grading` | 7. LLM-as-judge con rúbrica | Por respuesta | — |
| `final_assessment` | 7. Evaluación final | Por ruta e intento | — |
| `badge_text` | Badge | Por ruta | — |

Los prompts piden que el estudiante razone (preguntas de reflexión, retroalimentación que orienta sin dar la respuesta): el brief cita evidencia de que la IA sin salvaguardas puede dañar el aprendizaje (Bastani et al., 2025).

In [ ]:
PROMPTS = {
"syllabus_structuring": {"version": "v2.0",
 "system": "Eres un diseñador instruccional experto. Extraes la estructura de un syllabus universitario sin inventar contenidos que no estén en el texto. Respondes solo con JSON que cumple el esquema.",
 "user": """Estructura el siguiente syllabus.
Reglas:
- Un tema por unidad o bloque temático del syllabus. No agregues temas que no aparezcan.
- `id` en snake_case, corto y estable (ejemplo: regresion_logistica).
- De 2 a 4 resultados de aprendizaje (`learning_outcomes`) por tema, redactados en español como verbo de Bloom + contenido + criterio medible. `id` de cada resultado: lo_<id del tema>_<n>.
- `bloom_level` en inglés: remember, understand, apply, analyze, evaluate o create. El del tema es el nivel más alto entre sus resultados.
- `prerequisites` = ids de temas de ESTE syllabus que deben dominarse antes. Sin ciclos. Si el syllabus no los dice, dedúcelos solo cuando sean evidentes.
- `estimated_hours` = horas que indique el syllabus para ese tema; 0 si no lo indica. No las estimes.
- `default_objective`: el objetivo general del curso como resultado medible.
- `course_id` en snake_case. `name`, `description` y `subtopics` en español.

Syllabus:
<<<
{syllabus}
>>>"""},

"diagnostic_generation": {"version": "v2.0",
 "system": "Eres un experto en evaluación educativa y psicometría. Escribes ítems claros, con una sola respuesta correcta y distractores plausibles basados en errores frecuentes.",
 "user": """Genera una prueba diagnóstica para estimar el nivel previo de un estudiante en cada tema.
Para CADA tema escribe exactamente 3 preguntas de opción múltiple:
- una con `bloom_level` "remember" y `difficulty` 1,
- una con `bloom_level` "apply" y `difficulty` 2,
- una con `bloom_level` "analyze" y `difficulty` 3.
Cada pregunta: `stem` claro, 4 `options`, una sola correcta en `correct_option` (índice 0-based) y `rationale` que justifique la correcta y explique el error de cada distractor. Varía la posición de la correcta. `id` = d_<id del tema>_<n> con n = 1, 2, 3.
No uses "todas las anteriores" ni "ninguna de las anteriores". Textos en español.

Temas (JSON):
{topics}"""},

"item_review": {"version": "v1.0",
 "system": "Eres un revisor experto en evaluación educativa. Juzgas ítems con criterio estricto y no los corriges.",
 "user": """Revisa cada ítem y responde por cada uno usando su `question_id`:
- `single_correct_answer`: en opción múltiple, exactamente una opción es defendible como correcta y es la marcada en `correct_option`; en respuesta corta, hay una respuesta esperada clara que se puede calificar con la rúbrica.
- `clear_stem`: el enunciado se entiende sin ambigüedad y no da pistas de la respuesta.
- `bloom_matches`: la tarea cognitiva corresponde al `bloom_level` declarado.
- `aligned_with_outcome`: evalúa el resultado de aprendizaje indicado en `outcome`.
- `issues`: problemas concretos, en español (lista vacía si no hay).

Ítems (JSON):
{items}"""},

"simulated_student": {"version": "v2.0",
 "system": "Simulas a un estudiante real para una prueba de prototipo. Nunca respondes como un experto si el perfil no lo justifica.",
 "user": """Perfil del estudiante (en sus palabras):
<<<
{profile}
>>>

Lo que ha estudiado en la plataforma hasta ahora:
<<<
{context}
>>>

Responde las preguntas como lo haría ESTA persona: acierta lo que su formación y lo estudiado le permiten saber y equivócate de forma realista en lo que no domina (no aciertes todo por defecto).
- Opción múltiple: `selected_option` = índice 0-based, `dont_know` = false y `text` vacío.
- Respuesta corta: `selected_option` = -1, `dont_know` = false y `text` con 1 a 3 frases en el estilo de la persona.
- {dont_know_rule}
Responde todas las preguntas, en el mismo orden, usando su `id` como `question_id`.

Preguntas (JSON):
{questions}"""},

"learning_objective": {"version": "v2.0",
 "system": "Eres un tutor que redacta objetivos de aprendizaje medibles según la taxonomía de Bloom.",
 "user": """Propón el objetivo de aprendizaje personal de este estudiante para la ruta.
- Coherente con el objetivo de la ruta y centrado en su brecha (los temas con nivel bajo).
- `statement` en español: verbo de Bloom + contenido + criterio medible (ej.: "con al menos 80 % en la evaluación final"). `bloom_verb`, `content` y `criterion` son sus tres partes.
- `focus_topics`: ids de los temas de la brecha (solo ids de la lista).
- `rationale`: por qué este objetivo, en 1 o 2 frases.

Objetivo de la ruta: {path_objective}
Perfil del estudiante:
<<<
{profile}
>>>
Nivel por tema (0 = nada, 100 = domina): {levels}"""},

"topic_explanation": {"version": "v2.0",
 "system": "Eres un tutor universitario. Explicas usando SOLO la información de los fragmentos que se te entregan y citas cada afirmación con [n]. Haces pensar al estudiante en vez de darle respuestas listas.",
 "user": """Escribe la explicación principal del tema. Es un recurso compartido: la leerán todos los estudiantes de la ruta, así que no la adaptes a una persona en particular.
Tema (JSON): {topic}

Reglas:
- Usa SOLO los fragmentos. Cada afirmación factual lleva su cita [n]. No inventes fuentes ni números de cita.
- `sections`: 3 a 5 secciones que cubran los resultados de aprendizaje, en español claro.
- `worked_example`: un caso práctico genérico resuelto paso a paso.
- `summary`: 2 o 3 frases.
- `reflection_questions`: 2 preguntas que hagan razonar (sin respuesta).
- `gaps_detected`: resultados de aprendizaje que los fragmentos no cubren (lista vacía si no hay).

Fragmentos:
{fragments}"""},

"remedial_explanation": {"version": "v1.0",
 "system": "Eres un tutor universitario. Explicas usando SOLO la información de los fragmentos que se te entregan y citas cada afirmación con [n]. Haces pensar al estudiante en vez de darle respuestas listas.",
 "user": """El estudiante NO aprobó el quiz de este tema. Escribe una explicación ALTERNA, con otro enfoque (analogías y paso a paso) y centrada en sus errores. Esta explicación es personal: conecta los ejemplos con su perfil.
Tema (JSON): {topic}
Nivel estimado del estudiante en el tema: {level} (0 a 100).
Perfil:
<<<
{profile}
>>>
Preguntas que respondió mal: {errors}

Reglas:
- Usa SOLO los fragmentos y cita cada afirmación factual con [n]. No inventes fuentes.
- No des las respuestas de las preguntas falladas: explica el concepto para que pueda resolverlas.
- `sections`: 2 a 4 secciones. `worked_example`: un caso conectado con su perfil. `summary`, `reflection_questions` (2) y `gaps_detected` como en la explicación principal.

Fragmentos:
{fragments}"""},

"personalized_example": {"version": "v1.0",
 "system": "Eres un tutor que conecta los conceptos con la experiencia de cada estudiante, sin agregar contenido nuevo.",
 "user": """Escribe un ejemplo breve (4 a 6 frases) que aplique el tema a la experiencia de este estudiante. No agregues conceptos que no estén en los resultados de aprendizaje.
- `example`: el ejemplo, en español.
- `connection`: en una frase, por qué es relevante para esta persona.

Tema (JSON): {topic}
Perfil:
<<<
{profile}
>>>"""},

"rag_faithfulness": {"version": "v2.0",
 "system": "Eres un verificador de hechos estricto.",
 "user": """Descompón el texto en afirmaciones factuales atómicas (máximo 15). Para cada una: `text`, `citations` (números [n] que usa), `supported` (si los fragmentos citados de verdad la soportan) y `comment`. Ignora las frases que no son factuales.

Texto:
{text}

Fragmentos:
{fragments}"""},

"topic_quiz": {"version": "v2.0",
 "system": "Eres un experto en evaluación. Generas preguntas desde los resultados de aprendizaje, no desde un texto de estudio, para no evaluar solo memorización.",
 "user": """Genera el quiz del tema (intento {attempt}).
- {n_mc} preguntas de opción múltiple (`type` = "multiple_choice", 4 `options`, una correcta en `correct_option`, `rubric` vacía) y {n_sa} de respuesta corta (`type` = "short_answer", `options` vacío, `correct_option` = -1, `rubric` de 2 a 3 criterios cuyos `points` suman 1).
- Cubre todos los resultados de aprendizaje: `outcome_id` dice cuál evalúa cada pregunta y `bloom_level` corresponde a ese resultado. Mezcla niveles.
- `model_answer`: en opción múltiple, la justificación de la correcta; en respuesta corta, la respuesta esperada.
- `topic_id` = "{topic_id}". `id` = q<n>.
- Preguntas DISTINTAS a las de intentos anteriores, no reformulaciones. Textos en español.

Tema (JSON): {topic}
Preguntas de intentos anteriores (no repetir): {previous}"""},

"short_answer_grading": {"version": "v2.0",
 "system": "Eres un calificador justo y consistente. Aplicas la rúbrica literalmente, sin premiar la extensión ni penalizar la redacción.",
 "user": """Califica la respuesta del estudiante con la rúbrica. Por cada criterio: `criterion` (igual al de la rúbrica), `met`, `points_awarded` (entre 0 y los puntos del criterio; puedes dar puntaje parcial) y `evidence` (la parte de la respuesta que lo justifica). Luego `feedback`: retroalimentación breve, en español, que oriente sin dar la respuesta completa.

Pregunta: {question}
Respuesta esperada: {model_answer}
Rúbrica (JSON): {rubric}
Respuesta del estudiante:
<<<
{answer}
>>>"""},

"final_assessment": {"version": "v2.0",
 "system": "Eres un experto en evaluación. Diseñas evaluaciones sumativas alineadas con los resultados de aprendizaje.",
 "user": """Genera la evaluación final de la ruta (intento {attempt}).
- Por cada tema: 2 preguntas de opción múltiple (`type` = "multiple_choice", 4 `options`, una correcta, `rubric` vacía), una de nivel understand o apply y otra de nivel analyze.
- Además {n_sa} preguntas de respuesta corta que integren varios temas (`type` = "short_answer", `options` vacío, `correct_option` = -1, `rubric` de 2 a 3 criterios que suman 1). En `topic_id` va el tema principal.
- `outcome_id` = resultado de aprendizaje evaluado. `id` = f<n>. Usa solo `topic_id` de la lista.
- Preguntas distintas a las de intentos anteriores: {previous}
- Textos en español.

Temas de la ruta (JSON):
{topics}"""},

"badge_text": {"version": "v2.0",
 "system": "Redactas credenciales digitales sobrias y verificables.",
 "user": """Redacta el texto del badge en español: `name` corto, `description` de 1 o 2 frases, `criteria` de obtención (umbral y evaluaciones) y `competencies` con 3 a 5 competencias demostradas.
Curso: {course}
Objetivo logrado: {objective}
Temas de la ruta: {topics}
Umbral de la evaluación final: {threshold}"""},
}

for prompt_id, p in PROMPTS.items():
    (OUT / "prompts" / f"{prompt_id}.{p['version']}.json").write_text(
        json.dumps({"id": prompt_id, **p}, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(PROMPTS), "prompts versionados en", OUT / "prompts")

## 4. Capa de IA: llamada estructurada, embeddings, caché y registro

- `llm_json` exige el esquema con `response_json_schema`, valida con Pydantic y, si el JSON no cumple, reintenta mostrando el error. Cada intento queda en el log: de ahí sale la métrica del brief «tasa de salidas JSON inválidas < 2 %».
- **Caché por entrada:** la clave es el hash de modelo + prompt + versión + entrada. Como la explicación principal no lleva datos del estudiante, su clave es la misma para todos y se genera **una vez por tema**, como pide el brief.
- `embed` usa `task_type` distintos para documentos y consultas, y normaliza los vectores (necesario al reducir la dimensión).
- El log usa los campos de la tabla `llm_logs` de la API (`task`, `model`, `prompt_version`, `enrollment_id`, `input_tokens`, `output_tokens`, `cost_usd`, `latency_ms`, `status`, `error_message`), más `json_valid`, `call_attempt` e `input_sha256` para el análisis.

In [ ]:
def _now():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

def _sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def gemini_schema(model: type[BaseModel]) -> dict:
    """JSON Schema con las referencias $defs resueltas en línea."""
    s = model.model_json_schema()
    defs = s.pop("$defs", {})
    def resolve(x):
        if isinstance(x, dict):
            if "$ref" in x:
                return resolve(copy.deepcopy(defs[x["$ref"].split("/")[-1]]))
            return {k: resolve(v) for k, v in x.items()}
        if isinstance(x, list):
            return [resolve(i) for i in x]
        return x
    return resolve(s)

def _cost(model, t_in, t_out):
    p = PRICES_USD_PER_MILLION.get(model)
    return None if not p else (t_in * p["input"] + t_out * p["output"]) / 1e6

def _log(row: dict):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def llm_json(task: str, schema: type[BaseModel], variables: dict, temperature: float | None = None,
             max_retries: int = 2, enrollment_id: str | None = None):
    """Llama a Gemini con salida estructurada. Devuelve (objeto validado, Meta)."""
    p = PROMPTS[task]
    prompt = p["user"].format(**variables)
    key = _sha(json.dumps([LLM_MODEL, task, p["version"], p["system"], prompt, schema.__name__],
                          ensure_ascii=False))
    cache_path = OUT / "cache" / f"{task}_{key[:16]}.json"
    meta = dict(prompt_id=task, prompt_version=f"{task} {p['version']}", model=LLM_MODEL,
                generated_at=_now(), input_sha256=key)

    if cache_path.exists() and not REGENERATE:
        cached = json.loads(cache_path.read_text(encoding="utf-8"))
        meta.update(generated_at=cached["generated_at"], from_cache=True)
        return schema.model_validate(cached["output"]), Meta(**meta)

    contents = prompt
    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        error, obj, usage, text = None, None, None, ""
        try:
            resp = client.models.generate_content(
                model=LLM_MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=p["system"],
                    temperature=TEMPERATURE if temperature is None else temperature,
                    response_mime_type="application/json",
                    response_json_schema=gemini_schema(schema),
                ),
            )
            text = resp.text or ""
            usage = resp.usage_metadata
        except Exception as e:  # error de red o de la API: se registra y se reintenta
            error = f"API: {e}"
        latency = time.perf_counter() - t0
        t_in = (getattr(usage, "prompt_token_count", 0) or 0) if usage else 0
        t_out = ((getattr(usage, "candidates_token_count", 0) or 0) +
                 (getattr(usage, "thoughts_token_count", 0) or 0)) if usage else 0
        json_valid = None
        if error is None:
            try:
                obj = schema.model_validate_json(text)
                json_valid = True
            except ValidationError as e:
                error, json_valid = f"JSON inválido: {str(e)[:400]}", False
        _log({"ts": _now(), "task": task, "model": LLM_MODEL, "prompt_version": meta["prompt_version"],
              "enrollment_id": enrollment_id, "input_tokens": t_in, "output_tokens": t_out,
              "cost_usd": _cost(LLM_MODEL, t_in, t_out), "latency_ms": int(latency * 1000),
              "status": "success" if obj is not None else "error", "error_message": error,
              "json_valid": json_valid, "call_attempt": attempt + 1, "input_sha256": key})
        if obj is not None:
            cache_path.write_text(json.dumps({"generated_at": meta["generated_at"], "task": task,
                                              "prompt": prompt, "output": obj.model_dump()},
                                             ensure_ascii=False, indent=2), encoding="utf-8")
            return obj, Meta(**meta)
        print(f"  ⚠️ {task} intento {attempt + 1}: {error[:160]}")
        if json_valid is False:
            contents = prompt + f"\n\nTu respuesta anterior no cumplió el esquema ({error[:300]}). Devuelve solo JSON válido."
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"{task}: no se obtuvo una salida válida tras {max_retries + 1} intentos")

def embed(texts: list[str], task_type: str = "SEMANTIC_SIMILARITY") -> np.ndarray:
    """Embeddings normalizados (L2), con caché en disco."""
    key = _sha(json.dumps([EMBED_MODEL, EMBED_DIM, task_type, texts], ensure_ascii=False))
    cache_path = OUT / "cache" / f"emb_{key[:16]}.npy"
    if cache_path.exists() and not REGENERATE:
        return np.load(cache_path)
    vectors = []
    t0 = time.perf_counter()
    for i in range(0, len(texts), 100):
        r = client.models.embed_content(
            model=EMBED_MODEL, contents=texts[i:i + 100],
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM))
        vectors += [e.values for e in r.embeddings]
    V = np.array(vectors, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-12
    approx_tokens = sum(len(t) for t in texts) // 4
    _log({"ts": _now(), "task": f"embeddings:{task_type}", "model": EMBED_MODEL, "prompt_version": None,
          "enrollment_id": None, "input_tokens": approx_tokens, "output_tokens": 0,
          "cost_usd": _cost(EMBED_MODEL, approx_tokens, 0),
          "latency_ms": int((time.perf_counter() - t0) * 1000), "status": "success", "error_message": None,
          "json_valid": None, "call_attempt": 1, "input_sha256": key})
    np.save(cache_path, V)
    return V

def save(name: str, obj):
    data = obj.model_dump(by_alias=True) if isinstance(obj, BaseModel) else obj
    (OUT / "outputs" / f"{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def show(obj, n=1500):
    data = obj.model_dump(by_alias=True) if isinstance(obj, BaseModel) else obj
    s = json.dumps(data, ensure_ascii=False, indent=2)
    print(s[:n] + ("\n… (recortado)" if len(s) > n else ""))

# Prueba rápida de conexión
_r = client.models.generate_content(model=LLM_MODEL, contents="Responde solo: OK")
print("Gemini responde:", (_r.text or "").strip()[:40])

## 5. Insumos: syllabus, fuentes para el RAG y perfil de la estudiante

- **Syllabus.** El brief limita el MVP a 1 o 2 syllabus reales. Pon el del piloto en `syllabus.txt`, `.md` o `.pdf` junto al notebook y se carga solo; el de abajo es un ejemplo con la misma estructura del curso del prototipo (Fundamentos de ML).
- **Fuentes.** El texto se redacta solo con estos fragmentos (RAG), lo que reduce alucinaciones. Los de ejemplo son apuntes breves; para el piloto, pon libros abiertos o apuntes del docente en `fuentes/` (`.txt` o `.md`) y se trocean e indexan solos. Con pocas fuentes la explicación queda pobre y aparecen `gaps_detected`: eso también es un hallazgo de la prueba.
- **Estudiante.** «Laura» es ficticia y no hay datos personales reales (Ley 1581 de 2012). Su perfil es el mismo que muestra el prototipo.

In [ ]:
SYLLABUS_TEXT = """
MAESTRÍA EN CIENCIA DE DATOS — Curso: Fundamentos de Machine Learning (ruta corta, 4 semanas)

Objetivo general: que el estudiante construya, evalúe y compare modelos supervisados básicos sobre datos tabulares
y justifique la elección del modelo con métricas adecuadas.

Unidad 1. Fundamentos y preparación de datos. Aprendizaje supervisado vs. no supervisado, variables y etiquetas,
partición entrenamiento/prueba, fuga de datos, escalado de variables y codificación de variables categóricas.

Unidad 2. Regresión lineal. Modelo lineal, mínimos cuadrados, interpretación de coeficientes, error cuadrático medio
y R². Requiere la Unidad 1.

Unidad 3. Regresión logística. Clasificación binaria, función sigmoide, probabilidad y umbral de decisión,
log-loss, interpretación de coeficientes como log-odds y regularización L2. Requiere las Unidades 1 y 2.

Unidad 4. Árboles de decisión y ensambles. Criterio de división (Gini, entropía), profundidad y sobreajuste,
bosques aleatorios e importancia de variables. Requiere la Unidad 1.

Unidad 5. Evaluación de modelos de clasificación. Matriz de confusión, precisión, recall, F1, curva ROC y AUC,
validación cruzada k-fold y desbalance de clases. Requiere las Unidades 3 y 4.

Evaluación: quices por unidad (≥ 70 %) y evaluación final integradora (≥ 80 % para la insignia).
"""

for candidate in ["syllabus.txt", "syllabus.md", "syllabus.pdf"]:
    if pathlib.Path(candidate).exists():
        if candidate.endswith(".pdf"):
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
            from pypdf import PdfReader
            SYLLABUS_TEXT = "\n".join(pg.extract_text() or "" for pg in PdfReader(candidate).pages)
        else:
            SYLLABUS_TEXT = pathlib.Path(candidate).read_text(encoding="utf-8")
        print("Syllabus cargado desde", candidate)
        break

# Fuentes de ejemplo (apuntes del docente). url = referencia para ampliar.
SOURCES = [
 {"source_id": "apuntes_u1_a", "title": "Apuntes U1: aprendizaje supervisado", "url": "https://scikit-learn.org/stable/supervised_learning.html",
  "text": "En el aprendizaje supervisado cada ejemplo tiene variables de entrada y una etiqueta conocida; el modelo aprende una función que asigna entradas a etiquetas. En el no supervisado no hay etiquetas y se buscan estructuras, como grupos. Si la etiqueta es numérica continua el problema es de regresión; si es una categoría, de clasificación."},
 {"source_id": "apuntes_u1_b", "title": "Apuntes U1: partición y fuga de datos", "url": "https://scikit-learn.org/stable/common_pitfalls.html",
  "text": "Los datos se dividen en entrenamiento y prueba para estimar el desempeño en datos no vistos. La fuga de datos ocurre cuando información del conjunto de prueba o del futuro influye en el entrenamiento, por ejemplo al escalar con la media de todo el dataset antes de partir. Todo preprocesamiento debe ajustarse solo con el conjunto de entrenamiento."},
 {"source_id": "apuntes_u1_c", "title": "Apuntes U1: escalado y codificación", "url": "https://scikit-learn.org/stable/modules/preprocessing.html",
  "text": "La estandarización resta la media y divide por la desviación estándar de cada variable; es importante en modelos sensibles a la escala, como los regularizados. Las variables categóricas se convierten en números con codificación one-hot, que crea una columna binaria por categoría y evita imponer un orden falso."},
 {"source_id": "apuntes_u2_a", "title": "Apuntes U2: regresión lineal", "url": "https://scikit-learn.org/stable/modules/linear_model.html",
  "text": "La regresión lineal modela la variable respuesta como una combinación lineal de las variables de entrada más un intercepto. Los coeficientes se estiman por mínimos cuadrados, que minimizan la suma de los residuos al cuadrado. Cada coeficiente indica el cambio esperado en la respuesta por una unidad de cambio en esa variable, manteniendo las demás constantes."},
 {"source_id": "apuntes_u2_b", "title": "Apuntes U2: métricas de regresión", "url": "https://scikit-learn.org/stable/modules/model_evaluation.html",
  "text": "El error cuadrático medio (MSE) promedia los errores al cuadrado y penaliza más los errores grandes. El coeficiente R² mide la proporción de la varianza de la respuesta explicada por el modelo; vale 1 en un ajuste perfecto y puede ser negativo si el modelo es peor que predecir la media."},
 {"source_id": "apuntes_u3_a", "title": "Apuntes U3: regresión logística y sigmoide", "url": "https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression",
  "text": "La regresión logística es un modelo de clasificación binaria. Calcula una combinación lineal de las variables y la transforma con la función sigmoide, 1/(1+e^(-z)), que devuelve valores entre 0 y 1 interpretables como la probabilidad de la clase positiva. Para decidir la clase se compara esa probabilidad con un umbral, por defecto 0,5, que puede ajustarse según el costo de los errores."},
 {"source_id": "apuntes_u3_b", "title": "Apuntes U3: log-loss, log-odds y regularización", "url": "https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression",
  "text": "Los parámetros de la regresión logística se estiman minimizando la pérdida logarítmica (log-loss o entropía cruzada), que castiga con fuerza las predicciones seguras y equivocadas. Cada coeficiente es el cambio en el logaritmo de las odds de la clase positiva por unidad de la variable; exp(coeficiente) es la razón de odds. La regularización L2 agrega una penalización a los coeficientes grandes para reducir el sobreajuste; en scikit-learn su fuerza se controla con C, donde un C menor implica más regularización."},
 {"source_id": "apuntes_u4_a", "title": "Apuntes U4: árboles de decisión", "url": "https://scikit-learn.org/stable/modules/tree.html",
  "text": "Un árbol de decisión divide los datos con preguntas sobre las variables, eligiendo en cada nodo la división que más reduce la impureza, medida con el índice de Gini o la entropía. Los árboles profundos memorizan el entrenamiento y sobreajustan; limitar la profundidad o el número mínimo de muestras por hoja lo controla."},
 {"source_id": "apuntes_u4_b", "title": "Apuntes U4: bosques aleatorios", "url": "https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees",
  "text": "Un bosque aleatorio entrena muchos árboles sobre muestras bootstrap y con subconjuntos aleatorios de variables en cada división, y promedia sus predicciones; esto reduce la varianza frente a un solo árbol. La importancia de variables por reducción de impureza indica cuánto contribuye cada variable a las divisiones, aunque puede favorecer variables con muchas categorías."},
 {"source_id": "apuntes_u5_a", "title": "Apuntes U5: matriz de confusión y métricas", "url": "https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics",
  "text": "La matriz de confusión cuenta verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos. La precisión es VP/(VP+FP): de lo que el modelo marcó positivo, cuánto lo era. El recall es VP/(VP+FN): de los positivos reales, cuántos detectó. F1 es la media armónica de precisión y recall. Con clases desbalanceadas la exactitud puede ser engañosa."},
 {"source_id": "apuntes_u5_b", "title": "Apuntes U5: ROC, AUC y validación cruzada", "url": "https://scikit-learn.org/stable/modules/cross_validation.html",
  "text": "La curva ROC grafica la tasa de verdaderos positivos contra la de falsos positivos al variar el umbral; el AUC resume la curva y equivale a la probabilidad de que el modelo puntúe más alto a un positivo que a un negativo elegidos al azar. La validación cruzada k-fold divide el entrenamiento en k partes, entrena k veces dejando una parte para validar y promedia los resultados, lo que da una estimación más estable que una sola partición."},
]

# Fuentes propias en fuentes/*.txt|md → fragmentos de ~900 caracteres
for path in sorted(pathlib.Path("fuentes").glob("*")) if pathlib.Path("fuentes").exists() else []:
    if path.suffix.lower() in {".txt", ".md"}:
        parts = [p.strip() for p in re.split(r"\n\s*\n", path.read_text(encoding="utf-8")) if p.strip()]
        buf, k = "", 0
        for part in parts + [""]:
            if (len(buf) + len(part) > 900 or not part) and buf:
                k += 1
                SOURCES.append({"source_id": f"{path.stem}_{k}", "title": f"{path.stem} (fragmento {k})",
                                "url": path.name, "text": buf.strip()})
                buf = ""
            buf += "\n" + part
print(len(SOURCES), "fragmentos de fuentes")

# Estudiante ficticia (sin datos personales reales)
STUDENT = {
    "user_id": "u_laura",
    "name": "Laura Martínez (estudiante ficticia)",
    "email": "laura.ejemplo@example.org",
    "enrollment_id": "enr_laura_ml_path1",
    "profile": ("Soy ingeniera industrial y trabajo hace dos años como analista de datos en una empresa de logística. "
                "Uso Excel y SQL todos los días y hago tableros en Power BI. En la universidad vi estadística y regresión "
                "lineal, y este año hice un curso corto de Python con pandas. Nunca he entrenado un modelo de clasificación "
                "y no tengo claro cómo se evalúa un modelo más allá del R². Me interesa predecir qué pedidos llegarán tarde."),
}
LEARNING_PATH_ID = "path_ml_fundamentals_v1"
USE_PATH_OBJECTIVE = False  # True = usar el objetivo por defecto de la ruta sin pedirle uno al LLM

## 6. Paso 1 del mago: syllabus estructurado + validación del docente

**Brief, componente 1:** «Un LLM extrae del syllabus una lista de temas, subtemas, prerrequisitos y resultados de aprendizaje, con salida en JSON validada con un esquema. El resultado es un grafo dirigido de temas (DAG) que el docente revisa y aprueba. Sin esta validación humana, todo lo demás hereda los errores.»

El código verifica lo que el esquema no puede expresar: ids únicos, prerrequisitos que existen y ausencia de ciclos (la API aplica la misma regla al crear `topic_prerequisites`). Pantalla: **D2 · Curso y syllabus**.

**Qué observar en la prueba:** ¿los temas coinciden con las unidades? ¿Los resultados de aprendizaje son medibles? ¿El LLM inventó prerrequisitos u horas? Lo que el docente corrija abajo alimenta la métrica del brief «F1 de temas y prerrequisitos ≥ 0,85».

In [ ]:
syllabus_llm, meta_syl = llm_json("syllabus_structuring", SyllabusLLM, {"syllabus": SYLLABUS_TEXT}, temperature=0.1)

def topological_order(topics: list[SyllabusTopic]) -> list[str]:
    ids = [t.id for t in topics]
    assert len(ids) == len(set(ids)), f"ids de tema repetidos: {ids}"
    missing = {(t.id, p) for t in topics for p in t.prerequisites if p not in ids}
    assert not missing, f"prerrequisitos inexistentes: {missing}"
    pending = {t.id: set(t.prerequisites) - {t.id} for t in topics}
    order = []
    while pending:
        # El primer tema disponible en el orden del syllabus: respeta las unidades cuando el DAG lo permite
        ready = next((i for i in ids if i in pending and not pending[i]), None)
        assert ready, f"ciclo en los prerrequisitos: {pending}"
        order.append(ready)
        del pending[ready]
        for deps in pending.values():
            deps.discard(ready)
    return order

ORDER = topological_order(syllabus_llm.topics)
TOPICS = {t.id: t for t in syllabus_llm.topics}

display(pd.DataFrame([{
    "orden": ORDER.index(t.id) + 1, "id": t.id, "nombre": t.name, "bloom": BLOOM_LABEL_ES[t.bloom_level],
    "prerrequisitos": ", ".join(t.prerequisites) or "—", "resultados": len(t.learning_outcomes),
    "horas": t.estimated_hours or "no indica"} for t in syllabus_llm.topics]).sort_values("orden"))
print("Objetivo por defecto:", syllabus_llm.default_objective)

**Revisión del docente.** Si algo está mal, corrígelo en `TEACHER_CORRECTIONS` (por ejemplo `{"regresion_logistica": {"prerequisites": ["fundamentos", "regresion_lineal"]}}`), vuelve a correr la celda y deja `TEACHER_APPROVED = True` solo cuando el docente esté de acuerdo. Las correcciones quedan registradas en el contrato (`teacher_corrections`).

In [ ]:
TEACHER_CORRECTIONS = {}   # {topic_id: {campo: valor}}
TEACHER_APPROVED = True    # en la prueba real, solo después de que el docente revise la tabla

applied = []
for tid, changes in TEACHER_CORRECTIONS.items():
    for field, value in changes.items():
        applied.append(f"{tid}.{field}: {getattr(TOPICS[tid], field)!r} → {value!r}")
        setattr(TOPICS[tid], field, value)
ORDER = topological_order(list(TOPICS.values()))

syllabus = Syllabus(**syllabus_llm.model_dump(), approved_by_teacher=TEACHER_APPROVED,
                    teacher_corrections=applied, meta=meta_syl)
save("syllabus", syllabus)
print(f"{'✅ Aprobado' if TEACHER_APPROVED else '⏳ Pendiente de aprobación'}: {len(syllabus.topics)} temas, "
      f"{len(applied)} correcciones | prompt {meta_syl.prompt_version}")

## 7. Paso 2 del mago: diagnóstico + perfil semántico

**Brief, componentes 2 y 3:** «Para cada tema se generan 2 a 3 preguntas en tres niveles de Bloom (recordar, aplicar, analizar), con distractores plausibles y la respuesta correcta justificada. El nivel por tema se estima como el porcentaje de aciertos ponderado por dificultad.» «La descripción libre del estudiante se convierte en embeddings y se compara con los embeddings de cada tema [...] por ejemplo con un promedio ponderado donde la prueba pesa más.»

1. Gemini genera 3 preguntas por tema.
2. **Revisión automática de ítems** (`item_review`): un segundo prompt juzga si cada pregunta tiene una sola respuesta correcta, si el enunciado es claro, si el nivel de Bloom corresponde y si está alineada con el resultado de aprendizaje. Las preguntas sin una única respuesta correcta se descartan; las demás observaciones van a la revisión de expertos (§13). Es la mitigación del brief para «preguntas ambiguas o con varias respuestas correctas».
3. El estudiante simulado responde, con la opción «No lo sé» de la pantalla E3 (cuenta como no acierto). En el piloto, aquí van las respuestas reales.
4. El perfil se compara con cada tema y se combina con la prueba. El perfil pesa como máximo 25 % y menos si la persona escribió poco, para no penalizar a quien escribe menos (consideración ética del brief).
5. Clasificación por tema: **dominado** (≥ 80, se omite), **por reforzar** (50 a 79) o **por aprender** (< 50).

Pantallas: **E3 · Diagnóstico** y **E4 · Resultado del diagnóstico**.

In [ ]:
topics_json = json.dumps([{"id": t.id, "name": t.name, "subtopics": t.subtopics,
                           "learning_outcomes": [lo.statement for lo in t.learning_outcomes]}
                          for t in syllabus.topics], ensure_ascii=False)
diag_llm, meta_diag = llm_json("diagnostic_generation", DiagnosticLLM, {"topics": topics_json}, temperature=0.5)

# Validaciones que el esquema no puede expresar
problems = []
for q in diag_llm.questions:
    if q.topic_id not in TOPICS: problems.append(f"{q.id}: tema inexistente {q.topic_id}")
    if len(q.options) != 4: problems.append(f"{q.id}: {len(q.options)} opciones")
    if not 0 <= q.correct_option < len(q.options): problems.append(f"{q.id}: índice correcto fuera de rango")
    q.difficulty = min(3, max(1, q.difficulty))
diag_llm.questions = [q for q in diag_llm.questions
                      if q.topic_id in TOPICS and 0 <= q.correct_option < len(q.options)]
print("Preguntas por tema:", pd.Series([q.topic_id for q in diag_llm.questions]).value_counts().to_dict())
print("Problemas de forma:", problems or "ninguno")

def outcome_text(q) -> str:
    """Resultado de aprendizaje que evalúa la pregunta (o todos los del tema, si no lo indica)."""
    los = TOPICS[q.topic_id].learning_outcomes if q.topic_id in TOPICS else []
    oid = getattr(q, "outcome_id", None)
    match = [lo.statement for lo in los if lo.id == oid]
    return match[0] if match else " | ".join(lo.statement for lo in los)

def review_items(questions) -> tuple[list[ItemReview], list[str]]:
    """Revisión automática: devuelve las revisiones y los ids que se descartan."""
    items = [{"question_id": q.id, "type": getattr(q, "type", "multiple_choice"), "bloom_level": q.bloom_level,
              "stem": q.stem, "options": q.options, "correct_option": q.correct_option,
              "rubric": [c.model_dump() for c in getattr(q, "rubric", [])], "outcome": outcome_text(q)}
             for q in questions]
    r, _ = llm_json("item_review", ItemReviewLLM, {"items": json.dumps(items, ensure_ascii=False)}, temperature=0.0)
    by_id = {x.question_id: x for x in r.reviews}
    reviews = [by_id[q.id] for q in questions if q.id in by_id]
    discarded = [x.question_id for x in reviews if not x.single_correct_answer]
    flagged = [x for x in reviews if x.issues or not (x.clear_stem and x.bloom_matches and x.aligned_with_outcome)]
    print(f"  Revisión de ítems: {len(reviews)}/{len(questions)} revisados, {len(discarded)} descartados, "
          f"{len(flagged)} con observaciones")
    return reviews, discarded

diag_reviews, diag_discarded = review_items(diag_llm.questions)
diag_llm.questions = [q for q in diag_llm.questions if q.id not in diag_discarded]

def questions_for_student(questions):
    """Lo que ve la pantalla del estudiante: sin respuesta correcta, justificación ni rúbrica."""
    return [{"id": q.id, "stem": q.stem, "type": getattr(q, "type", "multiple_choice"), "options": q.options}
            for q in questions]

def simulate_student(questions, context: str, diagnostic=False):
    r, meta = llm_json("simulated_student", SimulatedAnswers, {
        "profile": STUDENT["profile"],
        "context": context or "Nada todavía (es el diagnóstico inicial).",
        "dont_know_rule": ("Es el diagnóstico inicial: si de verdad no sabe, puede elegir «No lo sé» "
                           "(`selected_option` = -1, `dont_know` = true)." if diagnostic else
                           "Es un quiz calificado: no existe la opción «No lo sé» (`dont_know` = false)."),
        "questions": json.dumps(questions_for_student(questions), ensure_ascii=False)},
        temperature=0.7, enrollment_id=STUDENT["enrollment_id"])
    by_id = {a.question_id: a for a in r.answers}
    # Toda pregunta queda con respuesta; si el simulador omitió alguna, cuenta como no respondida
    answers = [by_id.get(q.id, StudentAnswer(question_id=q.id, selected_option=-1, dont_know=False, text=""))
               for q in questions]
    if not diagnostic:
        for a in answers:
            a.dont_know = False
    return answers, meta

# --- Respuestas: estudiante simulado (en el piloto, las respuestas reales) ---
diag_answers, meta_diag_answers = simulate_student(diag_llm.questions, "", diagnostic=True)
answer_by_id = {a.question_id: a for a in diag_answers}
hits = {q.id: (not answer_by_id[q.id].dont_know) and answer_by_id[q.id].selected_option == q.correct_option
        for q in diag_llm.questions}

test_score = {}
for tid in TOPICS:
    qs = [q for q in diag_llm.questions if q.topic_id == tid]
    weight = sum(q.difficulty for q in qs)
    test_score[tid] = 100 * sum(q.difficulty * hits[q.id] for q in qs) / weight if weight else 0.0
    if not qs:
        print(f"⚠️ {tid} quedó sin preguntas válidas: su nivel sale solo del perfil")

# --- Perfil semántico ---
topic_texts = [f"{t.name}. {t.description} Subtemas: {', '.join(t.subtopics)}." for t in syllabus.topics]
E_topics = embed(topic_texts, "SEMANTIC_SIMILARITY")
E_profile = embed([STUDENT["profile"]], "SEMANTIC_SIMILARITY")[0]
sims = E_topics @ E_profile
spread = sims.max() - sims.min()
priors = 100 * (sims - sims.min()) / spread if spread > 1e-6 else np.zeros_like(sims)
words = len(STUDENT["profile"].split())
PROFILE_WEIGHT = 0.25 * min(1.0, words / 60)   # heurística a calibrar con los datos del piloto

def classify(level: float) -> str:
    if level >= MASTERY_SKIP_THRESHOLD: return "mastered"
    return "to_reinforce" if level >= REINFORCE_CUTOFF else "to_learn"

levels = []
for i, t in enumerate(syllabus.topics):
    combined = (1 - PROFILE_WEIGHT) * test_score[t.id] + PROFILE_WEIGHT * float(priors[i])
    levels.append(TopicLevel(topic_id=t.id, test_score=round(test_score[t.id], 1),
                             profile_similarity=round(float(sims[i]), 4), profile_prior=round(float(priors[i]), 1),
                             profile_weight=round(PROFILE_WEIGHT, 3), level=round(combined, 1),
                             classification=classify(combined), skip=combined >= MASTERY_SKIP_THRESHOLD))
LEVELS = {n.topic_id: n for n in levels}

total_weight = sum(q.difficulty for q in diag_llm.questions)
diagnostic = Diagnostic(
    enrollment_id=STUDENT["enrollment_id"], questions=diag_llm.questions, item_reviews=diag_reviews,
    discarded_questions=diag_discarded, answers=diag_answers, level_by_topic=levels,
    overall_score=round(100 * sum(q.difficulty * hits[q.id] for q in diag_llm.questions) / total_weight, 1),
    meta_questions=meta_diag, meta_answers=meta_diag_answers)
save("diagnostic", diagnostic)
display(pd.DataFrame([n.model_dump() for n in levels]))
print("Ejemplo de pregunta:"); show(diag_llm.questions[0], 900)

## 8. Paso 3 del mago: objetivo de aprendizaje

**Brief, componente 4:** «Si la ruta tiene un objetivo predefinido, se usa ese. Si no, el LLM propone uno redactado como resultado medible (verbo de Bloom + contenido + criterio), a partir del perfil y la brecha detectada. El estudiante lo confirma o ajusta.» Pantalla: **E4**, botón «Confirmar y ver mi ruta».

In [ ]:
gap = {n.topic_id: n.level for n in levels}
if USE_PATH_OBJECTIVE:
    objective = Objective(enrollment_id=STUDENT["enrollment_id"], origin="path_default", confirmed_by_student=True,
                          statement=syllabus.default_objective, bloom_verb="", content="", criterion="",
                          focus_topics=[n.topic_id for n in levels if not n.skip],
                          rationale="Objetivo por defecto de la ruta")
else:
    obj_llm, meta_obj = llm_json("learning_objective", ObjectiveLLM, {
        "path_objective": syllabus.default_objective, "profile": STUDENT["profile"],
        "levels": json.dumps(gap, ensure_ascii=False)}, enrollment_id=STUDENT["enrollment_id"])
    obj_llm.focus_topics = [t for t in obj_llm.focus_topics if t in TOPICS]
    objective = Objective(**obj_llm.model_dump(), enrollment_id=STUDENT["enrollment_id"], origin="llm_proposed",
                          confirmed_by_student=True,  # en la app, el estudiante lo confirma o lo edita
                          meta=meta_obj)
save("objective", objective)
print("🎯", objective.statement)
print("Temas foco:", objective.focus_topics)

## 9. Secuenciación de la ruta (determinista, sin LLM)

**Brief, componente 5:** «Se recorren los temas del DAG en orden topológico, se omiten los temas con dominio ≥ 80 % en el diagnóstico y se incluyen sus prerrequisitos débiles. Es un algoritmo determinista y explicable.»

Como todo tema por debajo del umbral queda incluido, los prerrequisitos débiles entran solos, y el orden topológico garantiza que se estudien antes. Los estados son los de `path_steps` en la API: `skipped` para los omitidos y `pending` para los demás. Pantallas: **E5 · Vista de ruta** y **D3 · Ruta y avance**.

In [ ]:
def sequence(levels: list[TopicLevel]) -> PersonalizedPath:
    level = {n.topic_id: n.level for n in levels}
    steps = []
    for k, tid in enumerate(ORDER, start=1):
        include = level[tid] < MASTERY_SKIP_THRESHOLD
        steps.append(PathStep(
            step_order=k, topic_id=tid, name=TOPICS[tid].name, included=include,
            reason=(f"nivel {level[tid]:.0f} < {MASTERY_SKIP_THRESHOLD}" if include
                    else f"dominado en el diagnóstico ({level[tid]:.0f})"),
            status="pending" if include else "skipped", score=None if include else level[tid]))
    return PersonalizedPath(enrollment_id=STUDENT["enrollment_id"], learning_path_id=LEARNING_PATH_ID,
                            algorithm="topological_threshold v1", mastery_skip_threshold=MASTERY_SKIP_THRESHOLD,
                            steps=steps)

path = sequence(levels)
save("learning_path", path)
display(pd.DataFrame([s.model_dump() for s in path.steps]))

## 10. Paso 4 del mago: un tema completo (texto con RAG, video y quiz)

**Brief, componentes 6 y 7:** «Para el texto se usa RAG: el LLM redacta la explicación del tema citando fragmentos de fuentes confiables [...] Para el video se buscan candidatos con la API de YouTube y se re-ordenan por similitud semántica [...] más filtros de duración e idioma.» «El quiz por tema y la evaluación final se generan desde los resultados de aprendizaje, no desde el texto del recurso, para evitar que evalúen solo memorización.»

**Qué es compartido y qué es personal.** El brief pide «guardar en caché el contenido generado por tema (no regenerar para cada estudiante)» y su modelo de datos dice que el recurso es «compartido entre estudiantes». Por eso:

| Contenido | Alcance | Prompt |
| --- | --- | --- |
| Explicación principal | Por tema, igual para todos (se genera una vez) | `topic_explanation` |
| Verificación de fidelidad | Por explicación | `rag_faithfulness` |
| Ejemplo conectado con el perfil | Por inscripción, corto y barato | `personalized_example` |
| Explicación alterna tras fallar el quiz | Por inscripción e intento | `remedial_explanation` |
| Video | Por tema | sin LLM (embeddings) |
| Quiz | Por tema e intento (preguntas nuevas en cada intento) | `topic_quiz` |

**Fidelidad.** Un segundo prompt separa la explicación en afirmaciones y marca cuáles soportan de verdad los fragmentos citados: es una aproximación a la métrica de fidelidad de RAGAS (meta del brief ≥ 0,8). Pantalla: **E6 · Tema**.

In [ ]:
E_sources = embed([f"{s['title']}. {s['text']}" for s in SOURCES], "RETRIEVAL_DOCUMENT")

def retrieve(topic: SyllabusTopic, k: int = 4):
    query = f"{topic.name}: {', '.join(topic.subtopics)}. " + " ".join(lo.statement for lo in topic.learning_outcomes)
    q = embed([query], "RETRIEVAL_QUERY")[0]
    s = E_sources @ q
    top = np.argsort(-s)[:k]
    return [(n + 1, SOURCES[i], float(s[i])) for n, i in enumerate(top)]

def fragments_text(recs):
    return "\n\n".join(f"[{n}] ({src['title']}) {src['text']}" for n, src, _ in recs)

def topic_json(topic):
    return json.dumps({"id": topic.id, "name": topic.name, "subtopics": topic.subtopics,
                       "learning_outcomes": [lo.model_dump() for lo in topic.learning_outcomes]}, ensure_ascii=False)

def explanation_text(e) -> str:
    return "\n".join([e.title] + [f"{s.heading}: {s.body}" for s in e.sections] + [e.worked_example, e.summary])

def _finish_explanation(e, meta, tid, kind, recs, enrollment_id=None):
    cited = {int(c) for c in re.findall(r"\[(\d+)\]", explanation_text(e))}
    invalid = sorted(c for c in cited if c < 1 or c > len(recs))
    uncited = [s.heading for s in e.sections if not re.search(r"\[\d+\]", s.body)]
    check, _ = llm_json("rag_faithfulness", FaithfulnessCheck, {
        "text": explanation_text(e), "fragments": fragments_text(recs)}, temperature=0.0)
    faithfulness = round(sum(c.supported for c in check.claims) / max(1, len(check.claims)), 3)
    unsupported = [c.text for c in check.claims if not c.supported]
    print(f"  {tid} [{kind}] citas={sorted(cited)} inválidas={invalid} secciones sin cita={uncited} "
          f"fidelidad={faithfulness}")
    return TopicExplanation(**e.model_dump(), topic_id=tid, kind=kind,
                            audience="all_students" if kind == "main" else "enrollment",
                            enrollment_id=enrollment_id, faithfulness=faithfulness,
                            unsupported_claims=unsupported, meta=meta,
                            sources=[CitedSource(n=n, source_id=s["source_id"], title=s["title"], url=s["url"])
                                     for n, s, _ in recs])

def main_explanation(tid: str):
    """Explicación principal: un recurso por tema, compartido por todos (no lleva datos del estudiante)."""
    recs = retrieve(TOPICS[tid])
    e, meta = llm_json("topic_explanation", TopicExplanationLLM, {
        "topic": topic_json(TOPICS[tid]), "fragments": fragments_text(recs)})
    return _finish_explanation(e, meta, tid, "main", recs), recs

def remedial_explanation(tid: str, errors: list[str]):
    """Explicación alterna tras fallar el quiz: personal (perfil, nivel y errores del estudiante)."""
    recs = retrieve(TOPICS[tid])
    e, meta = llm_json("remedial_explanation", TopicExplanationLLM, {
        "topic": topic_json(TOPICS[tid]), "level": LEVELS[tid].level, "profile": STUDENT["profile"],
        "errors": json.dumps(errors, ensure_ascii=False), "fragments": fragments_text(recs)},
        enrollment_id=STUDENT["enrollment_id"])
    return _finish_explanation(e, meta, tid, "remedial", recs, enrollment_id=STUDENT["enrollment_id"])

def personalized_example(tid: str) -> PersonalizedExample:
    x, meta = llm_json("personalized_example", PersonalizedExampleLLM, {
        "topic": topic_json(TOPICS[tid]), "profile": STUDENT["profile"]}, enrollment_id=STUDENT["enrollment_id"])
    return PersonalizedExample(**x.model_dump(), topic_id=tid, enrollment_id=STUDENT["enrollment_id"], meta=meta)

# Tema demo: regresión logística si está en la ruta; si no, el primer tema incluido
included = [s.topic_id for s in path.steps if s.included]
DEMO_TOPIC = next((t for t in included if "logist" in t), included[0] if included else ORDER[0])
explanation_demo, recs_demo = main_explanation(DEMO_TOPIC)
example_demo = personalized_example(DEMO_TOPIC)
save(f"topic_explanation_{DEMO_TOPIC}", explanation_demo)
save(f"personalized_example_{DEMO_TOPIC}", example_demo)

print("\n#", explanation_demo.title)
for s in explanation_demo.sections:
    print(f"\n## {s.heading}\n{textwrap.fill(s.body, 110)}")
print("\nEjemplo resuelto:", textwrap.fill(explanation_demo.worked_example, 110))
print("\nPara Laura:", textwrap.fill(example_demo.example, 110))
print("\nFuentes:", [(f.n, f.title) for f in explanation_demo.sources])
if explanation_demo.unsupported_claims:
    print("\nAfirmaciones no soportadas:", explanation_demo.unsupported_claims)

**Video.** Con `YOUTUBE_API_KEY` se buscan candidatos en español; sin ella, se usa la lista curada del docente (respaldo que propone el brief para «videos no disponibles o de baja calidad»). En ambos casos se filtran por duración y se re-ordenan por similitud de embeddings entre el título y la descripción del video y el tema. El brief propone comparar con la **transcripción**; agregarla es una mejora pendiente.

In [ ]:
import requests

YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not YOUTUBE_API_KEY:
    try:
        from google.colab import userdata
        YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
    except Exception:
        pass

# Lista curada del docente (respaldo): {topic_id: [{"video_id", "title", "channel", "description", "duration_min"}]}
TEACHER_VIDEOS = {}
DURATION_MIN, DURATION_MAX = 4, 30

def _iso8601_min(d):
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", d or "")
    return None if not m else int(m[1] or 0) * 60 + int(m[2] or 0) + int(m[3] or 0) / 60

def search_youtube(query, n=10):
    r = requests.get("https://www.googleapis.com/youtube/v3/search", timeout=20, params={
        "key": YOUTUBE_API_KEY, "part": "snippet", "q": query, "type": "video",
        "maxResults": n, "relevanceLanguage": "es", "videoEmbeddable": "true"})
    r.raise_for_status()
    items = r.json().get("items", [])
    ids = [it["id"]["videoId"] for it in items]
    durations = {}
    if ids:
        d = requests.get("https://www.googleapis.com/youtube/v3/videos", timeout=20, params={
            "key": YOUTUBE_API_KEY, "part": "contentDetails", "id": ",".join(ids)}).json()
        durations = {v["id"]: _iso8601_min(v["contentDetails"]["duration"]) for v in d.get("items", [])}
    return [{"video_id": it["id"]["videoId"], "title": it["snippet"]["title"], "channel": it["snippet"]["channelTitle"],
             "description": it["snippet"]["description"], "duration_min": durations.get(it["id"]["videoId"])}
            for it in items]

def video_resource(tid: str) -> VideoResource:
    topic = TOPICS[tid]
    if YOUTUBE_API_KEY:
        origin, cands = "youtube_api", search_youtube(f"{topic.name} machine learning explicación")
    elif TEACHER_VIDEOS.get(tid):
        origin, cands = "teacher_list", TEACHER_VIDEOS[tid]
    else:
        return VideoResource(topic_id=tid, origin="none", candidates=[], selected=None, embed_model=EMBED_MODEL)
    cands = [c for c in cands if c.get("duration_min") is None or DURATION_MIN <= c["duration_min"] <= DURATION_MAX]
    if not cands:
        return VideoResource(topic_id=tid, origin=origin, candidates=[], selected=None, embed_model=EMBED_MODEL)
    E_c = embed([f"{c['title']}. {c.get('description', '')}" for c in cands], "SEMANTIC_SIMILARITY")
    E_t = embed([f"{topic.name}. {topic.description}"], "SEMANTIC_SIMILARITY")[0]
    ranked = sorted(zip(cands, E_c @ E_t), key=lambda x: -x[1])
    candidates = [VideoCandidate(video_id=c["video_id"], title=c["title"], channel=c.get("channel", ""),
                                 url=f"https://www.youtube.com/watch?v={c['video_id']}",
                                 duration_min=c.get("duration_min"), similarity=round(float(s), 3))
                  for c, s in ranked]
    return VideoResource(topic_id=tid, origin=origin, candidates=candidates, selected=candidates[0],
                         embed_model=EMBED_MODEL)

video_demo = video_resource(DEMO_TOPIC)
save(f"video_{DEMO_TOPIC}", video_demo)
if video_demo.selected:
    display(pd.DataFrame([c.model_dump() for c in video_demo.candidates[:5]]))
else:
    print("⚠️ Sin YOUTUBE_API_KEY ni lista del docente: el tema queda sin video y E6 no se puede llenar completo.")

**Quiz.** Se genera desde los resultados de aprendizaje del tema; a diferencia de la v1, el prompt **no recibe el texto de estudio**. Cada intento pide preguntas distintas a las anteriores (regla del brief: «preguntas regeneradas en cada intento»). Antes de mostrarlo pasa por la revisión automática de ítems.

In [ ]:
N_MC, N_SA = 4, 1   # opción múltiple y respuesta corta por quiz

def _valid_question(q) -> bool:
    if q.type == "multiple_choice":
        return len(q.options) >= 2 and 0 <= q.correct_option < len(q.options)
    return bool(q.rubric)

def generate_quiz(tid: str, attempt: int, previous: list[str]) -> Quiz:
    q, meta = llm_json("topic_quiz", QuizLLM, {
        "attempt": attempt, "n_mc": N_MC, "n_sa": N_SA, "topic_id": tid, "topic": topic_json(TOPICS[tid]),
        "previous": json.dumps(previous, ensure_ascii=False) if previous else "ninguna"},
        temperature=0.6, enrollment_id=STUDENT["enrollment_id"])
    for p in q.questions:
        p.topic_id = tid
    questions = [p for p in q.questions if _valid_question(p)]
    reviews, discarded = review_items(questions)
    questions = [p for p in questions if p.id not in discarded]
    return Quiz(questions=questions, quiz_id=f"quiz_{tid}_a{attempt}", topic_id=tid, attempt=attempt,
                pass_threshold=TOPIC_PASS_THRESHOLD, item_reviews=reviews, discarded_questions=discarded, meta=meta)

quiz_demo = generate_quiz(DEMO_TOPIC, 1, [])
save(f"quiz_{DEMO_TOPIC}_a1", quiz_demo)
show(quiz_demo, 2500)

## 11. Calificación: regla para opción múltiple, LLM-as-judge para respuesta corta

**Brief, componente 7:** «Las respuestas cortas se califican con el LLM usando una rúbrica explícita (LLM-as-judge), y una muestra se califica también a mano para medir el acuerdo.» La muestra para calificar a mano se exporta en §13 (métrica: kappa ponderado cuadrático, QWK ≥ 0,6).

Escala de 0 a 100, como `score` en la API. Una respuesta corta cuenta como correcta con 60 o más. Pantallas: **E7 · Resultado del quiz** y **D4 · Detalle de estudiante**.

In [ ]:
def grade(assessment, answers, meta_answers: Meta, assessment_type="topic") -> Grading:
    by_id = {a.question_id: a for a in answers}
    results, metas = [], []
    for q in assessment.questions:
        a = by_id[q.id]
        if q.type == "multiple_choice":
            ok = (not a.dont_know) and a.selected_option == q.correct_option
            results.append(QuestionResult(question_id=q.id, topic_id=q.topic_id, type=q.type, correct=ok,
                                          score=100.0 if ok else 0.0, grader="rule",
                                          feedback="Correcto." if ok else q.model_answer))
            continue
        total = sum(c.points for c in q.rubric) or 1.0
        if not a.text.strip():
            results.append(QuestionResult(question_id=q.id, topic_id=q.topic_id, type=q.type, correct=False,
                                          score=0.0, grader="rule", feedback="Sin respuesta."))
            continue
        g, meta = llm_json("short_answer_grading", ShortAnswerGradingLLM, {
            "question": q.stem, "model_answer": q.model_answer,
            "rubric": json.dumps([c.model_dump() for c in q.rubric], ensure_ascii=False), "answer": a.text},
            temperature=0.0, enrollment_id=STUDENT["enrollment_id"])
        max_points = {c.criterion: c.points for c in q.rubric}
        awarded = sum(min(max(0.0, c.points_awarded), max_points.get(c.criterion, c.points_awarded)) for c in g.criteria)
        score = round(100 * min(1.0, awarded / total), 1)
        metas.append(meta)
        results.append(QuestionResult(question_id=q.id, topic_id=q.topic_id, type=q.type, correct=score >= 60,
                                      score=score, grader="llm_judge", feedback=g.feedback, rubric_detail=g.criteria))
    total_score = round(sum(r.score for r in results) / max(1, len(results)), 1)
    aid = assessment.quiz_id if assessment_type == "topic" else assessment.assessment_id
    return Grading(attempt_id=f"{aid}_{STUDENT['user_id']}", assessment_id=aid, assessment_type=assessment_type,
                   topic_id=getattr(assessment, "topic_id", None), answers=answers, results=results,
                   score=total_score, passed=total_score >= assessment.pass_threshold, submitted_at=_now(),
                   meta_answers=meta_answers, meta_grader=metas)

def studied_context(explanations: list[TopicExplanation], max_chars=3500):
    return "\n\n".join(explanation_text(e) for e in explanations)[-max_chars:]

answers_demo, meta_answers_demo = simulate_student(quiz_demo.questions, studied_context([explanation_demo]))
grading_demo = grade(quiz_demo, answers_demo, meta_answers_demo)
save(f"grading_{DEMO_TOPIC}_a1", grading_demo)
display(pd.DataFrame([{k: v for k, v in r.model_dump().items() if k != "rubric_detail"} for r in grading_demo.results]))
print(f"Puntaje: {grading_demo.score:.0f} → {'aprobado' if grading_demo.passed else 'no aprobado'} "
      f"(umbral {TOPIC_PASS_THRESHOLD})")

## 12. Recorrido completo de la ruta: bucle por tema, evaluación final y badge

Es el flujo del estudiante del brief: por cada tema incluido, explicación → quiz → calificación; con menos de 70, explicación alterna y quiz nuevo, hasta 3 intentos. Si agota los intentos, el paso queda en `failed` y se avisa al docente (estado «intentos agotados» del diseño). Con la ruta completa, evaluación final; con menos de 80, repaso dirigido de los temas débiles y nuevo intento; con 80 o más, badge Open Badges.

Las salidas del tema demo se reutilizan de la caché (mismo prompt y misma entrada → mismo contenido, como en producción). Todo queda en un solo estado del que salen las pantallas (§14). Pantallas: **E5** a **E12**.

In [ ]:
EVENTS, CONTENT, SNAPSHOTS = [], {}, {}
live_path = path.model_copy(deep=True)

def event(kind, **detail):
    EVENTS.append({"t": _now(), "type": kind, **detail})

event("enrolled", learning_path_id=LEARNING_PATH_ID)
event("diagnostic_completed", overall_score=diagnostic.overall_score)
event("objective_confirmed", objective=objective.statement)

studied = []
for step in live_path.steps:
    if not step.included:
        event("topic_skipped", topic_id=step.topic_id, level=step.score)
        continue
    tid = step.topic_id
    step.status = "in_progress"
    main, _ = main_explanation(tid)
    CONTENT[tid] = {"explanations": [main], "example": personalized_example(tid),
                    "video": video_resource(tid), "quizzes": [], "gradings": []}
    previous = []
    for attempt in range(1, MAX_ATTEMPTS + 1):
        quiz = generate_quiz(tid, attempt, previous)
        previous += [q.stem for q in quiz.questions]
        answers, meta_a = simulate_student(quiz.questions, studied_context(studied + CONTENT[tid]["explanations"]))
        g = grade(quiz, answers, meta_a)
        CONTENT[tid]["quizzes"].append(quiz)
        CONTENT[tid]["gradings"].append(g)
        step.attempts, step.score = attempt, g.score
        event("quiz_graded", topic_id=tid, attempt=attempt, score=g.score, passed=g.passed)
        print(f"{tid}: intento {attempt} → {g.score:.0f}")
        if g.passed:
            step.status = "passed"
            break
        errors = [q.stem for q in quiz.questions for r in g.results if r.question_id == q.id and r.score < 60]
        if attempt < MAX_ATTEMPTS:
            CONTENT[tid]["explanations"].append(remedial_explanation(tid, errors))
            event("remedial_delivered", topic_id=tid, attempt=attempt)
    else:
        step.status = "failed"
        event("attempts_exhausted", topic_id=tid)
    studied += CONTENT[tid]["explanations"]
    if tid == DEMO_TOPIC:
        SNAPSHOTS["in_progress"] = {"path": live_path.model_copy(deep=True), "n_events": len(EVENTS)}
    if step.status == "failed":
        print(f"⛔ {tid}: intentos agotados; requiere intervención del docente.")
        break
save("learning_path_final", live_path)

In [ ]:
final_assessment, final_grading, BADGE = None, None, None
path_topics = [s.topic_id for s in live_path.steps if s.included]
path_complete = all(s.status in ("passed", "skipped") for s in live_path.steps)
N_SA_FINAL = 2

if path_complete and path_topics:
    topics_final_json = json.dumps([json.loads(topic_json(TOPICS[t])) for t in path_topics], ensure_ascii=False)
    previous_final = []
    for attempt in range(1, MAX_ATTEMPTS + 1):
        fa, meta_fa = llm_json("final_assessment", QuizLLM, {
            "attempt": attempt, "n_sa": N_SA_FINAL, "topics": topics_final_json,
            "previous": json.dumps(previous_final, ensure_ascii=False) if previous_final else "ninguna"},
            temperature=0.6, enrollment_id=STUDENT["enrollment_id"])
        questions = [q for q in fa.questions if q.topic_id in path_topics and _valid_question(q)]
        reviews, discarded = review_items(questions)
        final_assessment = FinalAssessment(
            questions=[q for q in questions if q.id not in discarded], assessment_id=f"final_{LEARNING_PATH_ID}_a{attempt}",
            topics_assessed=path_topics, attempt=attempt, pass_threshold=FINAL_PASS_THRESHOLD,
            item_reviews=reviews, discarded_questions=discarded, meta=meta_fa)
        previous_final += [q.stem for q in final_assessment.questions]
        answers_f, meta_af = simulate_student(final_assessment.questions, studied_context(studied, 6000))
        final_grading = grade(final_assessment, answers_f, meta_af, assessment_type="final")
        save(f"final_assessment_a{attempt}", final_assessment)
        save(f"grading_final_a{attempt}", final_grading)
        event("final_graded", attempt=attempt, score=final_grading.score, passed=final_grading.passed)
        print(f"Evaluación final, intento {attempt}: {final_grading.score:.0f}")
        if final_grading.passed:
            break
        by_topic = pd.DataFrame([r.model_dump() for r in final_grading.results]).groupby("topic_id")["score"].mean()
        weak = [t for t, v in by_topic.items() if v < TOPIC_PASS_THRESHOLD and t in CONTENT]
        event("targeted_review", topics=weak)
        for t in weak:
            remedial = remedial_explanation(t, [f"Bajo desempeño en la evaluación final ({by_topic[t]:.0f})"])
            CONTENT[t]["explanations"].append(remedial)
            studied.append(remedial)
else:
    print("La ruta no está completa: no se habilita la evaluación final.")

def badge_svg(name: str) -> str:
    lines = textwrap.wrap(name, 18)[:3]
    text = "".join(f'<text x="120" y="{112 + i * 22}" text-anchor="middle" font-family="Arial" font-size="17" '
                   f'font-weight="bold" fill="#ffffff">{line}</text>' for i, line in enumerate(lines))
    return (f'<svg xmlns="http://www.w3.org/2000/svg" width="240" height="240" viewBox="0 0 240 240">'
            f'<circle cx="120" cy="120" r="112" fill="#1f4e79"/><circle cx="120" cy="120" r="98" fill="none" '
            f'stroke="#f2c14e" stroke-width="5"/><text x="120" y="78" text-anchor="middle" font-family="Arial" '
            f'font-size="13" fill="#f2c14e">SMARTMENTOR</text>{text}</svg>')

if final_grading and final_grading.passed:
    bt, meta_bt = llm_json("badge_text", BadgeTextLLM, {
        "course": syllabus.course_name, "objective": objective.statement,
        "topics": ", ".join(TOPICS[t].name for t in path_topics), "threshold": f"{FINAL_PASS_THRESHOLD} %"})
    (OUT / "outputs" / "badge.svg").write_text(badge_svg(bt.name), encoding="utf-8")
    salt = hashlib.sha256(STUDENT["enrollment_id"].encode()).hexdigest()[:12]
    BASE = "https://example.org/smartmentor"   # reemplazar por el dominio real de la plataforma
    BADGE = BadgeAssertion(
        id=f"{BASE}/assertions/{STUDENT['enrollment_id']}",
        recipient=OBRecipient(salt=salt, identity="sha256$" + hashlib.sha256((STUDENT["email"] + salt).encode()).hexdigest()),
        badge=OBBadgeClass(id=f"{BASE}/badges/{LEARNING_PATH_ID}", name=bt.name, description=bt.description,
                           image=f"{BASE}/badges/{LEARNING_PATH_ID}/badge.svg", criteria=OBCriteria(narrative=bt.criteria),
                           issuer=OBIssuer(id=f"{BASE}/issuer", name="SmartMentor (prototipo)", url=BASE),
                           tags=bt.competencies),
        issuedOn=_now(),
        evidence=[OBEvidence(narrative=f"Evaluación final {final_grading.score:.0f} (umbral {FINAL_PASS_THRESHOLD}); "
                                       f"prompt {meta_bt.prompt_version}")])
    save("badge", BADGE)
    event("badge_issued", badge_id=BADGE.id)
    show(BADGE)
    try:
        from IPython.display import SVG
        display(SVG(badge_svg(bt.name)))
    except Exception:
        pass
else:
    print("Sin badge: la evaluación final no alcanzó el umbral o no se habilitó.")

SNAPSHOTS["final"] = {"path": live_path.model_copy(deep=True), "n_events": len(EVENTS)}
SNAPSHOTS.setdefault("in_progress", SNAPSHOTS["final"])

## 13. Revisión humana: el juicio sobre el mago

La prueba sirve para **decidir si lo que genera la IA es útil** antes de construir el pipeline. Eso lo deciden personas, y el brief define cómo medirlo:

| Métrica del brief | Meta | Archivo que se exporta |
| --- | --- | --- |
| % de preguntas válidas (una sola correcta, claras, Bloom correcto), 2 expertos, acuerdo con kappa de Cohen | ≥ 80 % válidas, κ ≥ 0,6 | `review/items_for_experts.csv` |
| Acuerdo LLM contra humano en respuestas cortas (QWK) | ≥ 0,6 | `review/short_answers_for_human_grading.csv` |
| Fidelidad del texto a las fuentes | ≥ 0,8 | `review/explanations_for_review.csv` |

Cada archivo trae lo que dijo la revisión automática, para comparar. Los expertos llenan las columnas `expert1_*`, `expert2_*` y `human_score` (0 a 100) y se vuelve a ejecutar la celda siguiente, que calcula los acuerdos.

In [ ]:
REVIEW = OUT / "review"

def _item_rows(questions, reviews, source, attempt=None):
    rev = {r.question_id: r for r in reviews}
    for q in questions:
        r = rev.get(q.id)
        yield {"source": source, "attempt": attempt, "question_id": q.id, "topic_id": q.topic_id,
               "type": getattr(q, "type", "multiple_choice"), "bloom_level": q.bloom_level, "outcome": outcome_text(q),
               "stem": q.stem, "options": " | ".join(q.options),
               "correct_option": q.correct_option, "model_answer": getattr(q, "model_answer", getattr(q, "rationale", "")),
               "auto_single_correct": r.single_correct_answer if r else None,
               "auto_clear_stem": r.clear_stem if r else None, "auto_bloom_matches": r.bloom_matches if r else None,
               "auto_issues": "; ".join(r.issues) if r else "",
               "expert1_valid": "", "expert1_bloom_ok": "", "expert2_valid": "", "expert2_bloom_ok": "", "comments": ""}

items_path = REVIEW / "items_for_experts.csv"
if not items_path.exists():
    rows = list(_item_rows(diagnostic.questions, diagnostic.item_reviews, "diagnostic"))
    for tid, c in CONTENT.items():
        for quiz in c["quizzes"]:
            rows += list(_item_rows(quiz.questions, quiz.item_reviews, "topic_quiz", quiz.attempt))
    if final_assessment:
        rows += list(_item_rows(final_assessment.questions, final_assessment.item_reviews, "final", final_assessment.attempt))
    pd.DataFrame(rows).to_csv(items_path, index=False)

short_path = REVIEW / "short_answers_for_human_grading.csv"
if not short_path.exists():
    rows = []
    gradings = [g for c in CONTENT.values() for g in c["gradings"]] + ([final_grading] if final_grading else [])
    quizzes = {q.quiz_id: q for c in CONTENT.values() for q in c["quizzes"]}
    if final_assessment:
        quizzes[final_assessment.assessment_id] = final_assessment
    for g in gradings:
        qs = {q.id: q for q in quizzes[g.assessment_id].questions}
        ans = {a.question_id: a for a in g.answers}
        for r in g.results:
            if r.type == "short_answer":
                q = qs[r.question_id]
                rows.append({"assessment_id": g.assessment_id, "question_id": r.question_id, "stem": q.stem,
                             "model_answer": q.model_answer, "rubric": json.dumps([c.model_dump() for c in q.rubric], ensure_ascii=False),
                             "student_answer": ans[r.question_id].text, "llm_score": r.score, "human_score": ""})
    pd.DataFrame(rows).to_csv(short_path, index=False)

expl_path = REVIEW / "explanations_for_review.csv"
if not expl_path.exists():
    pd.DataFrame([{"topic_id": e.topic_id, "kind": e.kind, "audience": e.audience, "auto_faithfulness": e.faithfulness,
                   "unsupported_claims": " | ".join(e.unsupported_claims), "gaps_detected": " | ".join(e.gaps_detected),
                   "expert_faithful": "", "expert_clear": "", "comments": ""}
                  for c in CONTENT.values() for e in c["explanations"]]).to_csv(expl_path, index=False)
print("Archivos para revisión humana en", REVIEW.resolve())

def cohen_kappa(a, b, quadratic=False):
    a, b = np.asarray(a, dtype=int), np.asarray(b, dtype=int)
    cats = np.union1d(a, b)
    if len(cats) < 2:
        return float("nan")
    idx = {c: i for i, c in enumerate(cats)}
    O = np.zeros((len(cats), len(cats)))
    for x, y in zip(a, b):
        O[idx[x], idx[y]] += 1
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    k = len(cats)
    W = (np.subtract.outer(np.arange(k), np.arange(k)) ** 2) / (k - 1) ** 2 if quadratic else 1 - np.eye(k)
    return 1 - (W * O).sum() / (W * E).sum()

items = pd.read_csv(items_path)
done = items.dropna(subset=["expert1_valid", "expert2_valid"])
if len(done):
    e1, e2 = done["expert1_valid"].astype(int), done["expert2_valid"].astype(int)
    print(f"Ítems revisados por 2 expertos: {len(done)} | válidos (ambos): {(e1 & e2).mean():.0%} (meta ≥ 80 %) "
          f"| κ = {cohen_kappa(e1, e2):.2f} (meta ≥ 0,6)")
    print(f"Acuerdo revisión automática vs experto 1: {(done['auto_single_correct'].astype(bool) == e1.astype(bool)).mean():.0%}")
else:
    print("Pendiente: los expertos llenan expert1_valid y expert2_valid (1 = válida, 0 = no) en", items_path.name)

shorts = pd.read_csv(short_path)
graded = shorts.dropna(subset=["human_score"])
if len(graded) >= 2:
    to_band = lambda s: np.clip(np.round(np.asarray(s, dtype=float) / 25), 0, 4)   # 0-100 → 5 bandas
    print(f"Respuestas cortas calificadas a mano: {len(graded)} | QWK LLM vs humano = "
          f"{cohen_kappa(to_band(graded['llm_score']), to_band(graded['human_score']), quadratic=True):.2f} (meta ≥ 0,6)")
else:
    print("Pendiente: calificar a mano human_score (0-100) en", short_path.name)

## 14. Fuente única de datos → pantallas del prototipo

Todas las pantallas se derivan de **un solo JSON** (`outputs/single_source_laura.json`). Así se resuelven las inconsistencias del prototipo: el número de temas, el estado y el puntaje de cada tema y la fecha del badge salen del mismo dato en todas partes.

Se exportan dos escenarios internamente coherentes: `in_progress` (justo después del tema demo, para E5 «en curso») y `final` (ruta terminada, para E10 y E12). Las claves usan los códigos de las pantallas de Figma; `verify_consistency` comprueba las reglas que el análisis encontró rotas. Para llevarlos al prototipo: copiar los valores al Figma y a `app/src/clients/sample_seed.py`.

In [ ]:
SINGLE_SOURCE = {
    "data_version": "laura_example v2",
    "models": {"llm": LLM_MODEL, "embeddings": EMBED_MODEL},
    "course": {"id": syllabus.course_id, "name": syllabus.course_name},
    "learning_path": {"id": LEARNING_PATH_ID, "default_objective": syllabus.default_objective,
                      "topic_pass_threshold": TOPIC_PASS_THRESHOLD, "final_pass_threshold": FINAL_PASS_THRESHOLD,
                      "mastery_skip_threshold": MASTERY_SKIP_THRESHOLD, "max_attempts": MAX_ATTEMPTS},
    "student": {k: v for k, v in STUDENT.items() if k != "email"},
    "syllabus": syllabus.model_dump(),
    "diagnostic": diagnostic.model_dump(),
    "objective": objective.model_dump(),
    "content": {tid: {"explanations": [e.model_dump() for e in c["explanations"]],
                      "example": c["example"].model_dump(),
                      "video": c["video"].model_dump(),
                      "quizzes": [q.model_dump() for q in c["quizzes"]],
                      "gradings": [g.model_dump() for g in c["gradings"]]} for tid, c in CONTENT.items()},
    "final_assessment": final_assessment.model_dump() if final_assessment else None,
    "final_grading": final_grading.model_dump() if final_grading else None,
    "badge": BADGE.model_dump(by_alias=True) if BADGE else None,
    "events": EVENTS,
    "snapshots": {k: {"path": v["path"].model_dump(), "n_events": v["n_events"]} for k, v in SNAPSHOTS.items()},
}
save("single_source_laura", SINGLE_SOURCE)

def screens(F: dict, scenario: str) -> dict:
    path_s = F["snapshots"][scenario]["path"]
    events_s = F["events"][:F["snapshots"][scenario]["n_events"]]
    n_topics = len(F["syllabus"]["topics"])
    steps = path_s["steps"]
    done = [s for s in steps if s["status"] in ("passed", "skipped")]
    current = next((s for s in steps if s["status"] in ("in_progress", "pending", "failed")), None)
    badge_ok = any(e["type"] == "badge_issued" for e in events_s)
    demo = F["content"].get(DEMO_TOPIC)
    return {
        "E1_catalog": {"learning_path_id": F["learning_path"]["id"], "course": F["course"]["name"],
                       "num_topics": n_topics, "default_objective": F["learning_path"]["default_objective"]},
        "D2_syllabus_review": [{"id": t["id"], "name": t["name"], "bloom_level": t["bloom_level"],
                                "prerequisites": t["prerequisites"],
                                "learning_outcomes": [lo["statement"] for lo in t["learning_outcomes"]]}
                               for t in F["syllabus"]["topics"]],
        "E3_diagnostic": questions_for_student([DiagnosticQuestion(**q) for q in F["diagnostic"]["questions"]]),
        "E4_diagnostic_result": {"levels": [{"topic_id": n["topic_id"], "level": n["level"],
                                             "classification": n["classification"]} for n in F["diagnostic"]["level_by_topic"]],
                                 "proposed_objective": F["objective"]["statement"]},
        "E5_path_view": {"objective": F["objective"]["statement"], "total_topics": n_topics, "completed": len(done),
                         "current_topic": ({"topic_id": current["topic_id"], "position": current["step_order"],
                                            "label": f"Tema {current['step_order']} de {n_topics}"} if current else None),
                         "steps": [{"topic_id": s["topic_id"], "name": s["name"], "status": s["status"],
                                    "score": s["score"], "attempts": s["attempts"]} for s in steps],
                         "final_enabled": all(s["status"] in ("passed", "skipped") for s in steps)},
        "E6_topic": None if not demo else {
            "topic_id": DEMO_TOPIC, "explanation": demo["explanations"][0],
            "prompt_version": demo["explanations"][0]["meta"]["prompt_version"],
            "personalized_example": demo["example"]["example"],
            "video": demo["video"]["selected"],
            "quiz": {"quiz_id": demo["quizzes"][0]["quiz_id"], "prompt_version": demo["quizzes"][0]["meta"]["prompt_version"],
                     "questions": questions_for_student([QuizQuestion(**q) for q in demo["quizzes"][0]["questions"]])}},
        "E7_quiz_result": None if not demo else {k: demo["gradings"][0][k] for k in ("assessment_id", "score", "passed")},
        "D3_path_dashboard": [{"student": F["student"]["name"], "progress": f"{len(done)}/{n_topics}",
                               **{s["topic_id"]: s["score"] for s in steps}}],
        "D4_student_detail": [{"topic_id": e["topic_id"], "attempt": e["attempt"], "score": e["score"]}
                              for e in events_s if e["type"] == "quiz_graded"],
        "E10_badge": F["badge"] if badge_ok else None,
        "E12_achievements": [F["badge"]] if badge_ok else [],
    }

def verify_consistency(F: dict, P: dict) -> list[str]:
    errors = []
    n = len(F["syllabus"]["topics"])
    if P["E1_catalog"]["num_topics"] != n or len(P["D2_syllabus_review"]) != n or len(P["E5_path_view"]["steps"]) != n:
        errors.append("número de temas distinto entre catálogo, syllabus y vista de ruta")
    last = {}
    for row in P["D4_student_detail"]:
        last[row["topic_id"]] = row["score"]
    for s in P["E5_path_view"]["steps"]:
        if s["status"] == "skipped":
            continue
        if s["topic_id"] in last and last[s["topic_id"]] != s["score"]:
            errors.append(f"{s['topic_id']}: E5={s['score']} vs D4 último intento={last[s['topic_id']]}")
        if P["D3_path_dashboard"][0][s["topic_id"]] != s["score"]:
            errors.append(f"{s['topic_id']}: E5 y D3 no coinciden")
    if P["E10_badge"] and (not P["E5_path_view"]["final_enabled"] or P["E5_path_view"]["current_topic"]):
        errors.append("badge emitido sin terminar la ruta")
    if {x["topic_id"] for x in P["E4_diagnostic_result"]["levels"]} != {t["id"] for t in F["syllabus"]["topics"]}:
        errors.append("el diagnóstico no cubre exactamente los temas del syllabus")
    return errors

for scenario in ["in_progress", "final"]:
    P = screens(SINGLE_SOURCE, scenario)
    save(f"screens_{scenario}", P)
    errs = verify_consistency(SINGLE_SOURCE, P)
    print(f"Escenario {scenario}: {'✅ coherente' if not errs else '❌ ' + '; '.join(errs)}")
    print("   E5:", P["E5_path_view"]["current_topic"], "| completados", P["E5_path_view"]["completed"],
          "| badge:", "sí" if P["E10_badge"] else "no")

## 15. Validación de los contratos

Cada salida guardada se valida contra su esquema JSON exportado, igual que lo haría la integración continua del repositorio. Si una falla, el contrato y el código no coinciden y hay que corregirlo antes de versionar.

In [ ]:
import jsonschema

def validate(schema_name, data):
    schema = json.loads((OUT / "schemas" / f"{schema_name}.{SCHEMA_VERSION}.json").read_text(encoding="utf-8"))
    jsonschema.validate(data, schema)

checks = [("syllabus", syllabus), ("diagnostic", diagnostic), ("objective", objective),
          ("learning_path", live_path), ("topic_explanation", explanation_demo),
          ("personalized_example", example_demo), ("video_resource", video_demo),
          ("quiz", quiz_demo), ("grading", grading_demo)]
for c in CONTENT.values():
    checks += [("topic_explanation", e) for e in c["explanations"]] + [("quiz", q) for q in c["quizzes"]]
    checks += [("grading", g) for g in c["gradings"]] + [("personalized_example", c["example"]), ("video_resource", c["video"])]
if final_assessment: checks += [("final_assessment", final_assessment), ("grading", final_grading)]
if BADGE: checks.append(("badge", BADGE))

failures = 0
for name, obj in checks:
    try:
        validate(name, json.loads(obj.model_dump_json(by_alias=True)))
    except jsonschema.ValidationError as e:
        failures += 1
        print("❌", name, e.message[:200])
print(f"{len(checks) - failures}/{len(checks)} salidas cumplen su esquema")

## 16. Métricas de la prueba

Es una prueba de factibilidad con una estudiante simulada, no un resultado del piloto: sirve para ver si las salidas son útiles y cuánto cuestan antes de construir el pipeline. Se comparan con las metas del brief.

| Métrica del brief | Meta |
| --- | --- |
| Tasa de salidas JSON inválidas | < 2 % |
| Latencia p95 por generación | < 20 s |
| Costo por estudiante-ruta y por tema generado | Reportar |
| Fidelidad del texto (RAG) | ≥ 0,8 |
| % de ítems válidos (aquí, según la revisión automática; el dato oficial es el de los expertos) | ≥ 80 % |
| Temas omitidos frente a una ruta fija | Reportar |

La ganancia de Hake que se calcula al final es **solo ilustrativa**: el pre-test y el post-test no usan ítems equivalentes y las respuestas son simuladas. En el piloto hay que construir ítems paralelos.

In [ ]:
log = pd.read_json(LOG_PATH, lines=True)
summary = log.groupby("task").agg(
    calls=("task", "size"), invalid_json=("json_valid", lambda s: f"{(s == False).mean():.0%}"),
    input_tokens=("input_tokens", "sum"), output_tokens=("output_tokens", "sum"),
    latency_p50_ms=("latency_ms", "median"), latency_p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
    cost_usd=("cost_usd", lambda s: s.sum(min_count=1)))
display(summary)

gen = log[~log["task"].str.startswith("embeddings")]
print(f"Tasa de JSON inválido: {(gen['json_valid'] == False).mean():.1%} (meta < 2 %)")
print(f"Latencia p95 de generación: {gen['latency_ms'].quantile(0.95) / 1000:.1f} s (meta < 20 s)")
if log["cost_usd"].notna().any():
    per_student = log.loc[log["enrollment_id"].notna(), "cost_usd"].sum()
    shared = log.loc[log["enrollment_id"].isna(), "cost_usd"].sum()
    print(f"Costo total: USD {log['cost_usd'].sum():.4f} | por estudiante-ruta: USD {per_student:.4f} "
          f"| compartido (curso y temas): USD {shared:.4f}")
else:
    print("Costo: llena PRICES_USD_PER_MILLION en §1 para calcularlo.")

all_reviews = diagnostic.item_reviews + [r for c in CONTENT.values() for q in c["quizzes"] for r in q.item_reviews]
if final_assessment: all_reviews += final_assessment.item_reviews
main_faith = [e.faithfulness for c in CONTENT.values() for e in c["explanations"] if e.kind == "main" and e.faithfulness is not None]
attempts = [s.attempts for s in live_path.steps if s.included and s.attempts]
first_try = [c["gradings"][0].passed for c in CONTENT.values() if c["gradings"]]
qs_pre = [q for q in diagnostic.questions if q.topic_id in path_topics]
pre = (100 * sum(q.difficulty * hits[q.id] for q in qs_pre) / sum(q.difficulty for q in qs_pre)) if qs_pre else None
post = final_grading.score if final_grading else None
metrics = {
    "invalid_json_rate": round(float((gen["json_valid"] == False).mean()), 4),
    "latency_p95_s": round(float(gen["latency_ms"].quantile(0.95)) / 1000, 2),
    "rag_faithfulness_mean": round(float(np.mean(main_faith)), 3) if main_faith else None,
    "items_valid_auto": f"{np.mean([r.single_correct_answer for r in all_reviews]):.0%}" if all_reviews else None,
    "items_without_issues_auto": f"{np.mean([not r.issues for r in all_reviews]):.0%}" if all_reviews else None,
    "topics_skipped": sum(not s.included for s in live_path.steps),
    "first_attempt_pass_rate": f"{np.mean(first_try):.0%}" if first_try else None,
    "mean_attempts_per_topic": round(float(np.mean(attempts)), 2) if attempts else None,
    "pre_test_path_topics": round(pre, 1) if pre is not None else None,
    "post_test_final": post,
    "hake_gain_illustrative": round((post - pre) / (100 - pre), 3) if (pre is not None and post is not None and pre < 100) else None,
    "badge_issued": BADGE is not None,
}
save("test_metrics", metrics)
display(pd.Series(metrics, name="valor").to_frame())

## 17. Paquete para el repositorio y criterio de salida

El criterio de salida exige los esquemas **versionados en el repositorio**. Esta celda arma un paquete con la misma estructura del repo:

| Del notebook | Al repositorio |
| --- | --- |
| `schemas/*.v2.json` | `prompts/schemas/` |
| `prompts/*.json` | `prompts/` |
| `outputs/` y `llm_log.jsonl` | `data/examples/mago_de_oz/` |
| `review/*.csv` | `design/04-prueba-mago-de-oz/review/` |
| Este notebook, **guardado con sus salidas** | `design/04-prueba-mago-de-oz/` |

Si el notebook corre dentro del repositorio (en `design/04-prueba-mago-de-oz/`) y `COPY_TO_REPO = True`, copia directamente; si no (por ejemplo en Colab), descarga el zip.

In [ ]:
COPY_TO_REPO = False
PACKAGE = OUT / "package"
if PACKAGE.exists():
    shutil.rmtree(PACKAGE)
targets = {
    OUT / "schemas": PACKAGE / "prompts" / "schemas",
    OUT / "prompts": PACKAGE / "prompts",
    OUT / "outputs": PACKAGE / "data" / "examples" / "mago_de_oz",
    OUT / "review": PACKAGE / "design" / "04-prueba-mago-de-oz" / "review",
}
for src, dst in targets.items():
    shutil.copytree(src, dst, dirs_exist_ok=True)
shutil.copy(LOG_PATH, PACKAGE / "data" / "examples" / "mago_de_oz" / "llm_log.jsonl")

repo_root = pathlib.Path.cwd().parents[1] if len(pathlib.Path.cwd().parents) > 1 else None
if COPY_TO_REPO and repo_root and (repo_root / "prompts").is_dir():
    shutil.copytree(PACKAGE, repo_root, dirs_exist_ok=True)
    print("Copiado al repositorio en", repo_root)
else:
    zip_path = shutil.make_archive("mago_de_oz_smartmentor_v2", "zip", PACKAGE)
    print("Paquete:", zip_path)
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        pass

exit_criteria = {
    "Syllabus estructurado generado con el LLM": (OUT / "outputs" / "syllabus.json").exists(),
    "Diagnóstico generado con el LLM": (OUT / "outputs" / "diagnostic.json").exists(),
    "Tema completo: explicación y quiz": (OUT / "outputs" / f"quiz_{DEMO_TOPIC}_a1.json").exists(),
    "Tema completo: video": video_demo.selected is not None,
    "Esquemas exportados": len(list((OUT / "schemas").glob("*.json"))) == len(CONTRACTS),
    "Todas las salidas cumplen su esquema": failures == 0,
    "Pantallas coherentes desde una sola fuente": all(not verify_consistency(SINGLE_SOURCE, screens(SINGLE_SOURCE, s))
                                                      for s in ["in_progress", "final"]),
    "Revisión de expertos registrada": bool(len(done)),
}
for k, v in exit_criteria.items():
    print("✅" if v else "⏳", k)
print("\nFalta fuera del notebook: guardar este notebook con salidas, copiar el paquete al repo, "
      "cargar las pantallas en el prototipo y la aprobación del director.")